# Comprehensive Multinomial Error Analysis
##### Brain Connectivity Classification: 232 Regions Full Connectivity Model

## Setup and Data Loading

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import json
from pathlib import Path
import warnings
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
import pandas as pd
import json
from pathlib import Path
import warnings
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully")

Libraries imported successfully


In [3]:
# Set up paths
PROJECT_ROOT = Path('/home/sjoon/projects/brain_connectivity_classifier')
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'
ANALYSIS_OUTPUT_DIR = PROJECT_ROOT / 'data' / 'error_analysis' / 'multinomial'
ANALYSIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Create visualization output directory
VIZ_OUTPUT_DIR = ANALYSIS_OUTPUT_DIR / 'visualizations'
VIZ_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Results Directory: {RESULTS_DIR}")
print(f"Analysis Output: {ANALYSIS_OUTPUT_DIR}")
print(f"Visualization Output: {VIZ_OUTPUT_DIR}")

Results Directory: /home/sjoon/projects/brain_connectivity_classifier/data/results
Analysis Output: /home/sjoon/projects/brain_connectivity_classifier/data/error_analysis/multinomial
Visualization Output: /home/sjoon/projects/brain_connectivity_classifier/data/error_analysis/multinomial/visualizations


In [4]:
# Load helper functions
def load_json(filepath):
    """Load JSON file safely"""
    with open(filepath, 'r') as f:
        return json.load(f)

def load_npy(filepath):
    """Load numpy file safely"""
    return np.load(filepath, allow_pickle=True)

def load_csv(filepath):
    """Load CSV file safely"""
    return pd.read_csv(filepath)

print("Helper functions loaded")

Helper functions loaded


In [5]:
def create_region_info_from_metrics(per_region_metrics_df):
    """Create region_info DataFrame from per_region_metrics."""
    if per_region_metrics_df is None:
        return None
    
    # Make a copy to avoid modifying original
    df = per_region_metrics_df.copy()
    
    # Check if region_idx exists, if not create it
    if 'region_idx' not in df.columns:
        # Check if it's in the index
        if df.index.name == 'region_idx':
            df = df.reset_index()
        else:
            # Create region_idx from row number
            df['region_idx'] = range(len(df))
    
    # Select columns (only those that exist)
    columns_to_keep = ['region_idx']
    if 'region_name' in df.columns:
        columns_to_keep.append('region_name')
    if 'network' in df.columns:
        columns_to_keep.append('network')
    
    region_info = df[columns_to_keep].copy()
    
    # Add hemisphere information based on naming convention (if region_name exists)
    if 'region_name' in region_info.columns:
        region_info['hemisphere'] = region_info['region_name'].apply(
            lambda x: 'left' if x.startswith('LH_') or x.endswith('-lh') else 
                      'right' if x.startswith('RH_') or x.endswith('-rh') else 
                      'unknown'
        )
    
    return region_info

# --- LOAD FULL CONNECTIVITY ---
print(f"\n{'='*80}")
print("LOADING FULL CONNECTIVITY DATA")
print(f"{'='*80}")

full_cv = load_cv_bundle(full_multi_dir)
full_task = load_task_bundle(full_task_dir)

# Extract individual variables for backward compatibility
overall_metrics_full = full_cv['overall_metrics']
fold_metrics_full = full_cv['fold_metrics']
network_metrics_cv_full = full_cv['network_metrics']
per_region_metrics_cv_full = full_cv['per_region_metrics']
confusion_matrix_cv_full = full_cv['confusion_matrix']
cv_predictions_full = full_cv['cv_predictions']
cv_probabilities_full = full_cv['cv_probabilities']
cv_true_labels_full = full_cv['cv_true_labels']
cv_fold_indices_full = full_cv['cv_fold_indices']

task_summary_full = full_task['task_summary']
network_metrics_task_full = full_task['task_network_metrics']
per_region_metrics_task_full = full_task['task_per_region_metrics']
confusion_matrix_task_full = full_task['task_confusion_matrix']
task_predictions_full = full_task['task_predictions']
task_probabilities_full = full_task['task_probabilities']
task_true_labels_full = full_task['task_true_labels']

# Create region_info from per_region_metrics
region_info_full = create_region_info_from_metrics(per_region_metrics_cv_full)

print(f"✓ Full connectivity CV data loaded")
print(f"✓ Full connectivity task data loaded")
if region_info_full is not None:
    print(f"✓ Region info created: {len(region_info_full)} regions")

# --- LOAD LEFT HEMISPHERE ---
print(f"\n{'='*80}")
print("LOADING LEFT HEMISPHERE DATA")
print(f"{'='*80}")

lh_cv = load_cv_bundle(LH_multi_dir)
lh_task = load_task_bundle(LH_task_dir)

overall_metrics_lh = lh_cv['overall_metrics']
fold_metrics_lh = lh_cv['fold_metrics']
network_metrics_cv_lh = lh_cv['network_metrics']
per_region_metrics_cv_lh = lh_cv['per_region_metrics']
confusion_matrix_cv_lh = lh_cv['confusion_matrix']
cv_predictions_lh = lh_cv['cv_predictions']
cv_probabilities_lh = lh_cv['cv_probabilities']
cv_true_labels_lh = lh_cv['cv_true_labels']
cv_fold_indices_lh = lh_cv['cv_fold_indices']

task_summary_lh = lh_task['task_summary']
network_metrics_task_lh = lh_task['task_network_metrics']
per_region_metrics_task_lh = lh_task['task_per_region_metrics']
confusion_matrix_task_lh = lh_task['task_confusion_matrix']
task_predictions_lh = lh_task['task_predictions']
task_probabilities_lh = lh_task['task_probabilities']
task_true_labels_lh = lh_task['task_true_labels']

region_info_lh = create_region_info_from_metrics(per_region_metrics_cv_lh)

print(f"✓ Left hemisphere CV data loaded")
print(f"✓ Left hemisphere task data loaded")
if region_info_lh is not None:
    print(f"✓ Region info created: {len(region_info_lh)} regions")

# --- LOAD RIGHT HEMISPHERE ---
print(f"\n{'='*80}")
print("LOADING RIGHT HEMISPHERE DATA")
print(f"{'='*80}")

rh_cv = load_cv_bundle(RH_multi_dir)
rh_task = load_task_bundle(RH_task_dir)

overall_metrics_rh = rh_cv['overall_metrics']
fold_metrics_rh = rh_cv['fold_metrics']
network_metrics_cv_rh = rh_cv['network_metrics']
per_region_metrics_cv_rh = rh_cv['per_region_metrics']
confusion_matrix_cv_rh = rh_cv['confusion_matrix']
cv_predictions_rh = rh_cv['cv_predictions']
cv_probabilities_rh = rh_cv['cv_probabilities']
cv_true_labels_rh = rh_cv['cv_true_labels']
cv_fold_indices_rh = rh_cv['cv_fold_indices']

task_summary_rh = rh_task['task_summary']
network_metrics_task_rh = rh_task['task_network_metrics']
per_region_metrics_task_rh = rh_task['task_per_region_metrics']
confusion_matrix_task_rh = rh_task['task_confusion_matrix']
task_predictions_rh = rh_task['task_predictions']
task_probabilities_rh = rh_task['task_probabilities']
task_true_labels_rh = rh_task['task_true_labels']

region_info_rh = create_region_info_from_metrics(per_region_metrics_cv_rh)

print(f"✓ Right hemisphere CV data loaded")
print(f"✓ Right hemisphere task data loaded")
if region_info_rh is not None:
    print(f"✓ Region info created: {len(region_info_rh)} regions")


LOADING FULL CONNECTIVITY DATA


NameError: name 'load_cv_bundle' is not defined

In [ ]:
# 1. Clean and Parse Data
df = error_df.copy()
df.columns = ["Strategy", "Data", "N", "Rate", "SH_SN", "SH_DN", "DH_SN", "DH_DN"]

# Extract counts and percentages from strings like "167 (4.2%)"
cols_to_parse = ["SH_SN", "SH_DN", "DH_SN", "DH_DN"]
for col in cols_to_parse:
    df[[f"{col}_count", f"{col}_pct"]] = df[col].str.extract(r'(\d+) \(([\d.]+)%\)').astype(float)

# 2. Setup Plot
sns.set_style("white")
fig, ax = plt.subplots(figsize=(14, 8), dpi=100)

# Colors: Blues (Same Hemi), Oranges (Diff Hemi)
colors = ['#2b8cbe', '#a6bddb', '#e6550d', '#fdae6b']
labels = ["Same Hemi / Same Net", "Same Hemi / Diff Net", "Diff Hemi / Same Net", "Diff Hemi / Diff Net"]

# 3. Plot Stacked Bars
pct_cols = [f"{c}_pct" for c in cols_to_parse]
df_plot = df.set_index(['Strategy', 'Data'])[pct_cols]
df_plot.columns = labels
df_plot.plot(kind='bar', stacked=True, color=colors, ax=ax, width=0.7, edgecolor='black', linewidth=0.6)

# 4. Annotations
for i, (idx, row) in enumerate(df.iterrows()):
    cum_height = 0
    for j, col in enumerate(cols_to_parse):
        pct, count = row[f"{col}_pct"], int(row[f"{col}_count"])
        if pct > 2.0:  # Only label if visible
            ax.text(i, cum_height + (pct/2), f"{pct:.1f}%\n({count})", 
                    ha='center', va='center', color="white" if j in [0, 2] else "black", 
                    fontsize=9, fontweight='bold')
        cum_height += pct
    
    # Header labels (N and Rate)
    ax.text(i, 102, f"N={int(row['N'])}\n{row['Rate']}", ha='center', va='bottom', fontsize=10, fontweight='bold')

# 5. Visual Callouts & Polish
[ax.axvline(x, color='black', ls='--', lw=1, alpha=0.3) for x in [1.5, 3.5]]

# Bracket for Hemisphere-specific models
ax.plot([1.6, 1.6, 5.4, 5.4], [115, 117, 117, 115], color="black", lw=1.5)
ax.text(3.5, 118, "HEMISPHERE-RELATED ERRORS ELIMINATED", ha='center', fontweight='bold', color='#e6550d')

ax.set_ylabel("Proportion of Errors (%)", fontweight='bold')
ax.set_title("Impact of Hemisphere-Specific Constraints on Error Profiles", fontsize=15, pad=50, fontweight='bold')
ax.set_xticklabels([f"{s}\n{d}" for s, d in df_plot.index], rotation=0)
ax.set_ylim(0, 130)
ax.legend(title="Error Category", bbox_to_anchor=(1, 1), loc='upper left')
sns.despine()

plt.tight_layout()
plt.show()

In [6]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Define network colors (Yeo 7-network color scheme)
NETWORK_COLORS = {
    'Vis': 'rgb(120, 18, 134)',      # Visual - Purple
    'SomMot': 'rgb(70, 130, 180)',   # Somatomotor - Blue
    'DorsAttn': 'rgb(0, 118, 14)',   # Dorsal Attention - Green
    'SalVentAttn': 'rgb(196, 58, 250)', # Salience/Ventral Attention - Violet
    'Limbic': 'rgb(220, 248, 164)',  # Limbic - Cream
    'Cont': 'rgb(230, 148, 34)',     # Control - Orange
    'Default': 'rgb(205, 62, 78)',   # Default Mode - Red
    'Unknown': 'rgb(128, 128, 128)'  # Gray for unknown
}

def create_sankey_data_from_confusion(confusion_matrix, region_info, cv_predictions, cv_true_labels):
    """
    Create Sankey diagram data from confusion matrix and region info.
    
    Args:
        confusion_matrix: numpy array (n_regions x n_regions)
        region_info: DataFrame with columns ['region_idx', 'region_name', 'network', 'hemisphere']
        cv_predictions: numpy array of predicted region indices
        cv_true_labels: numpy array of true region indices
    
    Returns:
        DataFrame with columns: Source_Net, Source_Hemi, Target_Net, Target_Hemi, Value
    """
    if confusion_matrix is None or region_info is None:
        print("⚠️  Missing confusion matrix or region info")
        return pd.DataFrame()
    
    # Ensure region_info has hemisphere column
    if 'hemisphere' not in region_info.columns:
        region_info = region_info.copy()
        region_info['hemisphere'] = region_info['region_name'].apply(
            lambda x: 'Left' if x.startswith('LH_') or x.endswith('-lh') else 
                      'Right' if x.startswith('RH_') or x.endswith('-rh') else 
                      'Unknown'
        )
    
    # Create a mapping from region_idx to network and hemisphere
    region_to_net = dict(zip(region_info['region_idx'], region_info['network']))
    region_to_hemi = dict(zip(region_info['region_idx'], region_info['hemisphere']))
    
    # Build Sankey data
    sankey_rows = []
    
    n_regions = confusion_matrix.shape[0]
    for true_idx in range(n_regions):
        for pred_idx in range(n_regions):
            count = confusion_matrix[true_idx, pred_idx]
            
            if count == 0:
                continue
            
            # Get network and hemisphere info
            source_net = region_to_net.get(true_idx, 'Unknown')
            source_hemi = region_to_hemi.get(true_idx, 'Unknown')
            target_net = region_to_net.get(pred_idx, 'Unknown')
            target_hemi = region_to_hemi.get(pred_idx, 'Unknown')
            
            # Capitalize hemisphere names for consistency
            source_hemi = source_hemi.capitalize() if source_hemi != 'Unknown' else 'Unknown'
            target_hemi = target_hemi.capitalize() if target_hemi != 'Unknown' else 'Unknown'
            
            sankey_rows.append({
                'Source_Net': source_net,
                'Source_Hemi': source_hemi,
                'Target_Net': target_net,
                'Target_Hemi': target_hemi,
                'Value': int(count)
            })
    
    return pd.DataFrame(sankey_rows)

def to_rgba(rgb_str, alpha=0.5):
    """Converts an rgb string to rgba with specified alpha."""
    if 'rgba' in rgb_str: 
        return rgb_str
    return rgb_str.replace('rgb', 'rgba').replace(')', f', {alpha})')

def plot_multilevel_sankey_refined(df, title="Error Flow: Network Misclassifications"):
    if df.empty:
        print("⚠️  DataFrame is empty. Cannot create Sankey diagram.")
        return None

    # 1. Pre-calculate Error Counts per Network for the Dropdown
    # We only care about cases where the Source Network != Target Network
    error_df = df[df['Source_Net'] != df['Target_Net']]
    net_error_counts = error_df.groupby('Source_Net')['Value'].sum().to_dict()
    total_errors = sum(net_error_counts.values())

    # 2. Setup Data Structures
    nets = sorted(list(set(df['Source_Net']) | set(df['Target_Net'])))
    hemis = ["Left", "Right"]
    
    node_labels = nets + hemis + hemis + nets
    node_map = {
        'true':   {n: i for i, n in enumerate(nets)},
        'src_h':  {h: i + len(nets) for i, h in enumerate(hemis)},
        'pred_h': {h: i + len(nets) + 2 for i, h in enumerate(hemis)},
        'pred_n': {n: i + len(nets) + 4 for i, n in enumerate(nets)}
    }

    links = []

    # 3. Build the Flows
    # FLOW 1: True Network -> Source Hemi
    f1 = df.groupby(['Source_Net', 'Source_Hemi'])['Value'].sum().reset_index()
    for _, r in f1.iterrows():
        links.append(dict(
            source=node_map['true'][r['Source_Net']],
            target=node_map['src_h'][r['Source_Hemi']],
            value=r['Value'],
            color=to_rgba(NETWORK_COLORS.get(r['Source_Net'], 'rgb(128,128,128)'), 0.1),
            net_group=r['Source_Net']
        ))

    # FLOW 2: Source Hemi -> Predicted Hemi
    f2 = df.groupby(['Source_Net', 'Source_Hemi', 'Target_Hemi'])['Value'].sum().reset_index()
    for _, r in f2.iterrows():
        is_swap = r['Source_Hemi'] != r['Target_Hemi']
        links.append(dict(
            source=node_map['src_h'][r['Source_Hemi']],
            target=node_map['pred_h'][r['Target_Hemi']],
            value=r['Value'],
            color=to_rgba(NETWORK_COLORS.get(r['Source_Net'], 'rgb(128,128,128)'), 0.25 if is_swap else 0.05),
            net_group=r['Source_Net']
        ))

    # FLOW 3: Predicted Hemi -> Predicted Network (Error Focus)
    f3 = df.groupby(['Source_Net', 'Target_Hemi', 'Target_Net'])['Value'].sum().reset_index()
    for _, r in f3.iterrows():
        is_error = r['Source_Net'] != r['Target_Net']
        color = to_rgba(NETWORK_COLORS.get(r['Source_Net'], 'rgb(128,128,128)'), 0.8 if is_error else 0.05)
            
        links.append(dict(
            source=node_map['pred_h'][r['Target_Hemi']],
            target=node_map['pred_n'][r['Target_Net']],
            value=r['Value'],
            color=color,
            net_group=r['Source_Net']
        ))

    # 4. Create Figure
    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(
            pad=20, thickness=30,
            label=[l.upper() for l in node_labels],
            color=[NETWORK_COLORS.get(n, 'rgb(44,62,80)') for n in node_labels],
            line=dict(color="white", width=1)
        ),
        link=dict(
            source=[l['source'] for l in links],
            target=[l['target'] for l in links],
            value=[l['value'] for l in links],
            color=[l['color'] for l in links],
            customdata=[l['net_group'] for l in links],
            hovertemplate='<b>Source: %{customdata}</b><br>Target: %{target.label}<br>Count: %{value}<extra></extra>'
        )
    ))

    # 5. Interactive Dropdown Menu with Error Counts
    buttons = [dict(
        label=f"Show All Networks (Total Errors: {int(total_errors)})", 
        method="update", 
        args=[{"link.color": [l['color'] for l in links]}]
    )]

    for net in nets:
        # Get pre-calculated error count for this network
        err_count = int(net_error_counts.get(net, 0))
        
        focused_colors = []
        for l in links:
            if l['net_group'] == net:
                # Maintain original logic but ensure visibility
                focused_colors.append(l['color'].replace('0.05', '0.1').replace('0.8', '0.9'))
            else:
                focused_colors.append('rgba(0,0,0,0.01)') # Ghost the rest
        
        buttons.append(dict(
            label=f"Focus: {net} (Errors: {err_count})", 
            method="update", 
            args=[{"link.color": [focused_colors]}]
        ))

    fig.update_layout(
        title=dict(
            text=f"<b>{title}</b><br><span style='font-size:12px'>Vivid paths = misclassifications. Dropdown shows error counts per source.</span>", 
            x=0.5
        ),
        updatemenus=[dict(
            buttons=buttons, 
            direction="down", 
            showactive=True, 
            x=0, y=1.15, 
            xanchor='left'
        )],
        font_family="Arial",
        height=850,
        margin=dict(t=150, b=50, l=50, r=50),
        paper_bgcolor='white'
    )

    return fig

# ============================================================================
# USAGE EXAMPLES
# ============================================================================

# Example 1: Full Connectivity Sankey
print("\n" + "="*80)
print("CREATING SANKEY DIAGRAMS")
print("="*80)

print("\nGenerating Full Connectivity Sankey data...")
sankey_data_full = create_sankey_data_from_confusion(
    confusion_matrix_cv_full,
    region_info_full,
    cv_predictions_full,
    cv_true_labels_full
)
print(f"✓ Created {len(sankey_data_full)} flow entries")

if not sankey_data_full.empty:
    fig_full = plot_multilevel_sankey_refined(
        sankey_data_full, 
        title="Full Brain (232 Regions) - Error Flow Analysis"
    )
    if fig_full:
        fig_full.write_html(VIZ_OUTPUT_DIR / 'sankey_full_connectivity.html')
        print(f"✓ Saved: {VIZ_OUTPUT_DIR / 'sankey_full_connectivity.html'}")
        fig_full.show()

# Example 2: Left Hemisphere Sankey
print("\nGenerating Left Hemisphere Sankey data...")
sankey_data_lh = create_sankey_data_from_confusion(
    confusion_matrix_cv_lh,
    region_info_lh,
    cv_predictions_lh,
    cv_true_labels_lh
)
print(f"✓ Created {len(sankey_data_lh)} flow entries")

if not sankey_data_lh.empty:
    fig_lh = plot_multilevel_sankey_refined(
        sankey_data_lh, 
        title="Left Hemisphere (116 Regions) - Error Flow Analysis"
    )
    if fig_lh:
        fig_lh.write_html(VIZ_OUTPUT_DIR / 'sankey_left_hemisphere.html')
        print(f"✓ Saved: {VIZ_OUTPUT_DIR / 'sankey_left_hemisphere.html'}")
        fig_lh.show()

# Example 3: Right Hemisphere Sankey
print("\nGenerating Right Hemisphere Sankey data...")
sankey_data_rh = create_sankey_data_from_confusion(
    confusion_matrix_cv_rh,
    region_info_rh,
    cv_predictions_rh,
    cv_true_labels_rh
)
print(f"✓ Created {len(sankey_data_rh)} flow entries")

if not sankey_data_rh.empty:
    fig_rh = plot_multilevel_sankey_refined(
        sankey_data_rh, 
        title="Right Hemisphere (116 Regions) - Error Flow Analysis"
    )
    if fig_rh:
        fig_rh.write_html(VIZ_OUTPUT_DIR / 'sankey_right_hemisphere.html')
        print(f"✓ Saved: {VIZ_OUTPUT_DIR / 'sankey_right_hemisphere.html'}")
        fig_rh.show()

print("\n" + "="*80)
print("SANKEY DIAGRAMS COMPLETE")
print("="*80)


CREATING SANKEY DIAGRAMS

Generating Full Connectivity Sankey data...


NameError: name 'confusion_matrix_cv_full' is not defined

In [7]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================================
# 1. CONFIGURATION & COLOR MAPPING
# ============================================================================
NETWORK_COLORS = {
    'Visual': 'rgb(120, 18, 134)', 
    'Somatomotor': 'rgb(70, 130, 180)',
    'Dorsal Attention': 'rgb(0, 118, 14)', 
    'Salience/Ventral Attention': 'rgb(196, 58, 250)',
    'Limbic': 'rgb(220, 248, 164)', 
    'Control': 'rgb(230, 148, 34)',
    'Default': 'rgb(205, 62, 78)', 
    'Subcortical': 'rgb(12, 123, 174)', 
    'Other': 'rgb(128, 128, 128)'
}

def to_rgba(rgb_str, alpha=0.3):
    return rgb_str.replace('rgb', 'rgba').replace(')', f', {alpha})')

# ============================================================================
# 2. DATA EXTRACTION FUNCTION
# ============================================================================
def extract_sankey_data(cm, region_info):
    """Extract error flow data from confusion matrix"""
    rows = []
    
    # Determine hemisphere
    if 'hemisphere' in region_info.columns:
        is_left = region_info['hemisphere'].str.lower() == 'left'
    else:
        is_left = region_info['region_name'].str.startswith(('LH_', 'L_')) | \
                  region_info['region_name'].str.contains('_L')
    
    region_info = region_info.reset_index(drop=True)
    
    for i in range(cm.shape[0]):
        if i >= len(region_info):
            continue
            
        true_net = region_info.iloc[i]['major_network']
        true_hemi = "Left" if is_left.iloc[i] else "Right"
        
        for j in range(cm.shape[1]):
            if i == j or cm[i, j] == 0:
                continue
                
            pred_net = region_info.iloc[j]['major_network']
            pred_hemi = "Left" if is_left.iloc[j] else "Right"
            
            rows.append({
                'Source_Net': true_net,
                'Source_Hemi': true_hemi,
                'Target_Hemi': pred_hemi,
                'Target_Net': pred_net,
                'Value': cm[i, j]
            })
    
    return pd.DataFrame(rows)

# ============================================================================
# 3. CALCULATE KEY METRICS
# ============================================================================
def calculate_metrics(df):
    """Calculate key error pattern metrics"""
    if df.empty:
        return {}
    
    total_errors = df['Value'].sum()
    
    # Inter-hemispheric errors (crossing errors)
    inter_hemi = df[df['Source_Hemi'] != df['Target_Hemi']]['Value'].sum()
    inter_hemi_pct = (inter_hemi / total_errors) * 100 if total_errors > 0 else 0
    
    # Intra-hemispheric errors (same hemisphere)
    intra_hemi = df[df['Source_Hemi'] == df['Target_Hemi']]['Value'].sum()
    intra_hemi_pct = (intra_hemi / total_errors) * 100 if total_errors > 0 else 0
    
    # Network-level errors
    network_errors = df.groupby('Source_Net')['Value'].sum().sort_values(ascending=False)
    top_error_network = network_errors.index[0] if len(network_errors) > 0 else "N/A"
    top_error_pct = (network_errors.iloc[0] / total_errors) * 100 if len(network_errors) > 0 else 0
    
    # Cross-network errors (different networks)
    cross_network = df[df['Source_Net'] != df['Target_Net']]['Value'].sum()
    cross_network_pct = (cross_network / total_errors) * 100 if total_errors > 0 else 0
    
    return {
        'total_errors': int(total_errors),
        'inter_hemi': int(inter_hemi),
        'inter_hemi_pct': inter_hemi_pct,
        'intra_hemi': int(intra_hemi),
        'intra_hemi_pct': intra_hemi_pct,
        'top_error_network': top_error_network,
        'top_error_pct': top_error_pct,
        'cross_network_pct': cross_network_pct
    }

# ============================================================================
# 4. ENHANCED SANKEY TRACE GENERATOR
# ============================================================================
def create_sankey_trace(df, domain_dict, show_percentages=True):
    """Generates a single Sankey trace with enhanced flow tracking"""
    if df.empty:
        return None
    
    all_networks = sorted(set(df['Source_Net'].unique()) | set(df['Target_Net'].unique()))
    source_hemis = sorted(df['Source_Hemi'].unique())
    target_hemis = sorted(df['Target_Hemi'].unique())
    
    nodes = []
    node_colors = []
    
    # Tier 0: True Networks
    for n in all_networks:
        nodes.append(f"True {n}")
        node_colors.append(NETWORK_COLORS.get(n, NETWORK_COLORS['Other']))
    
    # Tier 1: Source Hemispheres
    for h in source_hemis:
        nodes.append(f"Src {h}")
        node_colors.append('rgb(100, 100, 100)')
    
    # Tier 2: Target Hemispheres
    for h in target_hemis:
        nodes.append(f"Pred {h}")
        node_colors.append('rgb(100, 100, 100)')
    
    # Tier 3: Predicted Networks
    for n in all_networks:
        nodes.append(f"Predicted {n}")
        node_colors.append(NETWORK_COLORS.get(n, NETWORK_COLORS['Other']))
    
    node_map = {name: i for i, name in enumerate(nodes)}
    links = []
    total_val = df['Value'].sum()

    # Flow 1: True Net -> Source Hemi
    f1 = df.groupby(['Source_Net', 'Source_Hemi'])['Value'].sum().reset_index()
    for _, r in f1.iterrows():
        pct = (r['Value'] / total_val) * 100
        links.append(dict(
            source=node_map[f"True {r['Source_Net']}"], 
            target=node_map[f"Src {r['Source_Hemi']}"],
            value=r['Value'], 
            color=to_rgba(NETWORK_COLORS.get(r['Source_Net'], NETWORK_COLORS['Other'])),
            label=f"{pct:.1f}%" if show_percentages else ""
        ))

    # Flow 2: Source Hemi -> Pred Hemi (KEY FLOW - Inter-hemispheric crossing)
    f2 = df.groupby(['Source_Hemi', 'Target_Hemi'])['Value'].sum().reset_index()
    for _, r in f2.iterrows():
        pct = (r['Value'] / total_val) * 100
        # Highlight inter-hemispheric crossings with different color
        is_crossing = r['Source_Hemi'] != r['Target_Hemi']
        color = 'rgba(255, 100, 100, 0.5)' if is_crossing else 'rgba(150, 150, 150, 0.4)'
        
        links.append(dict(
            source=node_map[f"Src {r['Source_Hemi']}"], 
            target=node_map[f"Pred {r['Target_Hemi']}"],
            value=r['Value'], 
            color=color,
            label=f"{pct:.1f}%" if show_percentages else ""
        ))

    # Flow 3: Pred Hemi -> Predicted Net
    f3 = df.groupby(['Target_Hemi', 'Target_Net'])['Value'].sum().reset_index()
    for _, r in f3.iterrows():
        pct = (r['Value'] / total_val) * 100
        links.append(dict(
            source=node_map[f"Pred {r['Target_Hemi']}"], 
            target=node_map[f"Predicted {r['Target_Net']}"],
            value=r['Value'], 
            color=to_rgba(NETWORK_COLORS.get(r['Target_Net'], NETWORK_COLORS['Other'])),
            label=f"{pct:.1f}%" if show_percentages else ""
        ))

    return go.Sankey(
        domain=domain_dict,
        arrangement="perpendicular",
        node=dict(
            pad=20, 
            thickness=20, 
            line=dict(color="black", width=0.5),
            label=[n.split(' ')[-1] for n in nodes], 
            color=node_colors
        ),
        link=dict(
            source=[l['source'] for l in links], 
            target=[l['target'] for l in links],
            value=[l['value'] for l in links], 
            color=[l['color'] for l in links],
            customdata=[l.get('label', "") for l in links],
            hovertemplate='<b>Errors:</b> %{value}<br><b>% of Total:</b> %{customdata}<extra></extra>'
        )
    )

# ============================================================================
# 5. FIGURE 1: TWO-PANEL COMPARISON (Rest vs Task)
# ============================================================================

def create_figure1_comparison(confusion_matrix_cv_full, confusion_matrix_task_full, 
                               region_info_full):
    """
    Creates Figure 1: Detailed comparison of Rest vs Task error patterns
    for Full Strategy only (your main thesis finding)
    """
    
    # Extract data for both conditions
    df_rest = extract_sankey_data(confusion_matrix_cv_full, region_info_full)
    df_task = extract_sankey_data(confusion_matrix_task_full, region_info_full)
    
    # Calculate metrics
    metrics_rest = calculate_metrics(df_rest)
    metrics_task = calculate_metrics(df_task)
    
    # Calculate change
    inter_hemi_change = metrics_task['inter_hemi_pct'] - metrics_rest['inter_hemi_pct']
    
    # Create figure
    fig = go.Figure()
    
    # Panel A: Resting State (Left side)
    trace_rest = create_sankey_trace(
        df_rest,
        {'x': [0.02, 0.48], 'y': [0.15, 0.85]}
    )
    if trace_rest is not None:
        fig.add_trace(trace_rest)
    
    # Panel B: Task State (Right side)
    trace_task = create_sankey_trace(
        df_task,
        {'x': [0.52, 0.98], 'y': [0.15, 0.85]}
    )
    if trace_task is not None:
        fig.add_trace(trace_task)
    
    # Create annotations
    annotations = []
    
    # Panel titles
    annotations.extend([
        dict(x=0.25, y=0.95, text="<b>A) Resting State (Training Data)</b>",
             showarrow=False, xref="paper", yref="paper", 
             font=dict(size=16, color="black")),
        dict(x=0.75, y=0.95, text="<b>B) Task State (Test Data)</b>",
             showarrow=False, xref="paper", yref="paper", 
             font=dict(size=16, color="black"))
    ])
    
    # Tier labels for Panel A
    tier_labels = ["TRUE NET", "ORIGIN", "PRED", "PRED NET"]
    for i, label in enumerate(tier_labels):
        annotations.append(dict(
            x=0.02 + i * 0.115,
            y=0.88,
            text=f"<i>{label}</i>",
            showarrow=False,
            xref="paper",
            yref="paper",
            font=dict(size=10, color="gray")
        ))
    
    # Tier labels for Panel B
    for i, label in enumerate(tier_labels):
        annotations.append(dict(
            x=0.52 + i * 0.115,
            y=0.88,
            text=f"<i>{label}</i>",
            showarrow=False,
            xref="paper",
            yref="paper",
            font=dict(size=10, color="gray")
        ))
    
    # Metrics boxes for Panel A (Rest)
    annotations.extend([
        dict(x=0.25, y=0.08, 
             text=f"<b>Total Errors:</b> {metrics_rest['total_errors']}<br>" +
                  f"<b>Inter-hemispheric:</b> {metrics_rest['inter_hemi_pct']:.1f}%<br>" +
                  f"<b>Intra-hemispheric:</b> {metrics_rest['intra_hemi_pct']:.1f}%<br>" +
                  f"<b>Top Error Network:</b> {metrics_rest['top_error_network']} ({metrics_rest['top_error_pct']:.1f}%)",
             showarrow=False, xref="paper", yref="paper",
             font=dict(size=11), align="center",
             bgcolor="rgba(240, 240, 240, 0.8)",
             bordercolor="black", borderwidth=1)
    ])
    
    # Metrics boxes for Panel B (Task)
    change_symbol = "↑" if inter_hemi_change > 0 else "↓"
    change_color = "red" if inter_hemi_change > 0 else "green"
    
    annotations.extend([
        dict(x=0.75, y=0.08,
             text=f"<b>Total Errors:</b> {metrics_task['total_errors']}<br>" +
                  f"<b>Inter-hemispheric:</b> {metrics_task['inter_hemi_pct']:.1f}% " +
                  f"<span style='color:{change_color}'>{change_symbol}{abs(inter_hemi_change):.1f}pp</span><br>" +
                  f"<b>Intra-hemispheric:</b> {metrics_task['intra_hemi_pct']:.1f}%<br>" +
                  f"<b>Top Error Network:</b> {metrics_task['top_error_network']} ({metrics_task['top_error_pct']:.1f}%)",
             showarrow=False, xref="paper", yref="paper",
             font=dict(size=11), align="center",
             bgcolor="rgba(240, 240, 240, 0.8)",
             bordercolor="black", borderwidth=1)
    ])
    
    # Add interpretation box
    annotations.append(dict(
        x=0.5, y=0.01,
        text="<i>Red flows indicate inter-hemispheric crossings | Gray flows indicate intra-hemispheric errors</i>",
        showarrow=False, xref="paper", yref="paper",
        font=dict(size=10, color="gray"), align="center"
    ))
    
    fig.update_layout(
        title_text="<b>Figure 1: Brain Network Classification Error Flow - Rest vs Task Comparison</b><br>" +
                   "<sup>Full-Brain Training Strategy | Error-as-Signal Framework</sup>",
        height=900,
        width=1600,
        annotations=annotations,
        margin=dict(t=100, b=100, l=40, r=40),
        paper_bgcolor='white',
        font=dict(family="Arial, sans-serif")
    )
    
    return fig, metrics_rest, metrics_task

# ============================================================================
# EXECUTION
# ============================================================================

# Generate Figure 1
fig1, metrics_rest, metrics_task = create_figure1_comparison(
    confusion_matrix_cv_full, 
    confusion_matrix_task_full, 
    region_info_full
)

fig1.show()

# Optional: Save
# fig1.write_html("figure1_rest_vs_task_comparison.html")
# fig1.write_image("figure1_rest_vs_task_comparison.png", width=1600, height=900, scale=2)

print("\n" + "="*60)
print("FIGURE 1 METRICS SUMMARY")
print("="*60)
print(f"\nResting State:")
print(f"  Total Errors: {metrics_rest['total_errors']}")
print(f"  Inter-hemispheric: {metrics_rest['inter_hemi_pct']:.2f}%")
print(f"  Top Error Network: {metrics_rest['top_error_network']}")

print(f"\nTask State:")
print(f"  Total Errors: {metrics_task['total_errors']}")
print(f"  Inter-hemispheric: {metrics_task['inter_hemi_pct']:.2f}%")
print(f"  Top Error Network: {metrics_task['top_error_network']}")

print(f"\nChange (Task vs Rest):")
print(f"  Inter-hemispheric: {metrics_task['inter_hemi_pct'] - metrics_rest['inter_hemi_pct']:+.2f} pp")
print("="*60)

NameError: name 'confusion_matrix_cv_full' is not defined

The inter-hemisphere error decreased during task slightly by 2.33 percentage points, the absolute number of inter-hemispheric erros actually increased by 425 errors. Intra hemispheric increased even more by 625 errors -- which causes relative decrease. This suggests that the Gender Stroop task induces greater within-hemisphere network confusion rather than primarily disrupting cross-hemispheric communication. This pattern may reflect task-specific hemispheric specialization, where each hemisphere increases its internal processing load, leading to more localized network overlap and confusion. This finding actually supports the error-as-signal framework—the task reveals a specific type of functional reorganization: increased within-hemisphere processing demands."

In [ ]:
# Extract Resting State data
df_rest = extract_sankey_data(confusion_matrix_cv_full, region_info_full)
metrics_rest = calculate_metrics(df_rest)

fig = go.Figure()

# Single-panel Resting State Sankey
trace_rest = create_sankey_trace(
    df_rest,
    {'x': [0.02, 0.98], 'y': [0.15, 0.85]}  # use full width
)
if trace_rest is not None:
    fig.add_trace(trace_rest)

annotations = []

# Panel title
annotations.append(dict(
    x=0.5, y=0.95,
    text="<b>Resting State (Training Data)</b>",
    showarrow=False, xref="paper", yref="paper",
    font=dict(size=16, color="black")
))

# Tier labels (now just once across the whole figure)
tier_labels = ["TRUE NET", "ORIGIN", "PRED", "PRED NET"]
for i, label in enumerate(tier_labels):
    annotations.append(dict(
        x=0.02 + i * 0.32,  # spread across 0–1
        y=0.88,
        text=f"<i>{label}</i>",
        showarrow=False,
        xref="paper",
        yref="paper",
        font=dict(size=10, color="gray")
    ))

# Metrics box for Rest
annotations.append(dict(
    x=0.5, y=0.08,
    text=(
        f"<b>Total Errors:</b> {metrics_rest['total_errors']}<br>"
        f"<b>Inter-hemispheric:</b> {metrics_rest['inter_hemi_pct']:.1f}%<br>"
        f"<b>Intra-hemispheric:</b> {metrics_rest['intra_hemi_pct']:.1f}%<br>"
        f"<b>Top Error Network:</b> {metrics_rest['top_error_network']} "
        f"({metrics_rest['top_error_pct']:.1f}%)"
    ),
    showarrow=False, xref="paper", yref="paper",
    font=dict(size=11), align="center",
    bgcolor="rgba(240, 240, 240, 0.8)",
    bordercolor="black", borderwidth=1
))

# Interpretation box
annotations.append(dict(
    x=0.5, y=0.01,
    text="<i>Red flows indicate inter-hemispheric crossings | Gray flows indicate intra-hemispheric errors</i>",
    showarrow=False, xref="paper", yref="paper",
    font=dict(size=10, color="gray"), align="center"
))

fig.update_layout(
    title_text="<b>Brain Network Classification Error Flow - Resting State</b><br>"
               "<sup>Full-Brain Training Strategy | Error-as-Signal Framework</sup>",
    height=900,
    width=1200,
    annotations=annotations,
    margin=dict(t=100, b=100, l=40, r=40),
    paper_bgcolor='white',
    font=dict(family="Arial, sans-serif")
)

fig.show()


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================================
# 1. CONFIGURATION
# ============================================================================
NETWORK_COLORS = {
    'Visual': 'rgb(120, 18, 134)', 
    'Somatomotor': 'rgb(70, 130, 180)',
    'Dorsal Attention': 'rgb(0, 118, 14)', 
    'Salience/Ventral Attention': 'rgb(196, 58, 250)',
    'Limbic': 'rgb(220, 248, 164)', 
    'Control': 'rgb(230, 148, 34)',
    'Default': 'rgb(205, 62, 78)', 
    'Subcortical': 'rgb(12, 123, 174)', 
    'Other': 'rgb(128, 128, 128)'
}

# ============================================================================
# 2. DATA EXTRACTION (REUSE FROM FIGURE 1)
# ============================================================================
def extract_sankey_data(cm, region_info):
    """Extract error flow data from confusion matrix"""
    rows = []
    
    if 'hemisphere' in region_info.columns:
        is_left = region_info['hemisphere'].str.lower() == 'left'
    else:
        is_left = region_info['region_name'].str.startswith(('LH_', 'L_')) | \
                  region_info['region_name'].str.contains('_L')
    
    region_info = region_info.reset_index(drop=True)
    
    for i in range(cm.shape[0]):
        if i >= len(region_info):
            continue
            
        true_net = region_info.iloc[i]['major_network']
        true_hemi = "Left" if is_left.iloc[i] else "Right"
        
        for j in range(cm.shape[1]):
            if i == j or cm[i, j] == 0:
                continue
                
            pred_net = region_info.iloc[j]['major_network']
            pred_hemi = "Left" if is_left.iloc[j] else "Right"
            
            rows.append({
                'Source_Net': true_net,
                'Source_Hemi': true_hemi,
                'Target_Hemi': pred_hemi,
                'Target_Net': pred_net,
                'Value': cm[i, j]
            })
    
    return pd.DataFrame(rows)

# ============================================================================
# 3. COMPREHENSIVE METRICS EXTRACTION
# ============================================================================
def extract_all_metrics(mapping):
    """Extract metrics for all strategies and conditions"""
    
    results = []
    
    for (strategy, condition), (cm, region_info) in mapping.items():
        df = extract_sankey_data(cm, region_info)
        
        if df.empty:
            continue
        
        total_errors = df['Value'].sum()
        
        # Inter-hemispheric errors
        inter_hemi = df[df['Source_Hemi'] != df['Target_Hemi']]['Value'].sum()
        inter_hemi_pct = (inter_hemi / total_errors) * 100 if total_errors > 0 else 0
        
        # Intra-hemispheric errors
        intra_hemi = df[df['Source_Hemi'] == df['Target_Hemi']]['Value'].sum()
        intra_hemi_pct = (intra_hemi / total_errors) * 100 if total_errors > 0 else 0
        
        # Left-to-Right and Right-to-Left
        left_to_right = df[(df['Source_Hemi'] == 'Left') & (df['Target_Hemi'] == 'Right')]['Value'].sum()
        right_to_left = df[(df['Source_Hemi'] == 'Right') & (df['Target_Hemi'] == 'Left')]['Value'].sum()
        
        l2r_pct = (left_to_right / total_errors) * 100 if total_errors > 0 else 0
        r2l_pct = (right_to_left / total_errors) * 100 if total_errors > 0 else 0
        
        results.append({
            'Strategy': strategy,
            'Condition': condition,
            'Total_Errors': int(total_errors),
            'Inter_Hemi_Pct': inter_hemi_pct,
            'Intra_Hemi_Pct': intra_hemi_pct,
            'L2R_Pct': l2r_pct,
            'R2L_Pct': r2l_pct
        })
    
    return pd.DataFrame(results)

def extract_network_metrics(mapping):
    """Extract network-specific error patterns"""
    
    results = []
    
    for (strategy, condition), (cm, region_info) in mapping.items():
        df = extract_sankey_data(cm, region_info)
        
        if df.empty:
            continue
        
        total_errors = df['Value'].sum()
        
        # Network-wise errors (as source of error)
        network_errors = df.groupby('Source_Net')['Value'].sum()
        
        for network, errors in network_errors.items():
            pct = (errors / total_errors) * 100 if total_errors > 0 else 0
            
            results.append({
                'Strategy': strategy,
                'Condition': condition,
                'Network': network,
                'Error_Count': int(errors),
                'Error_Pct': pct
            })
    
    return pd.DataFrame(results)

# ============================================================================
# 4. FIGURE 2: THREE-PANEL COMPARATIVE ANALYSIS
# ============================================================================

def create_figure2_comparative(mapping):
    """
    Creates Figure 2: Quantitative comparison across all strategies
    Panel A: Inter-hemispheric error rates
    Panel B: Network-specific error contributions
    Panel C: Strategy performance comparison
    """
    
    # Extract all metrics
    df_metrics = extract_all_metrics(mapping)
    df_network = extract_network_metrics(mapping)
    
    # Create subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "<b>A) Inter-Hemispheric Error Rates</b>",
            "<b>B) Hemispheric Asymmetry (L→R vs R→L)</b>",
            "<b>C) Network Error Contributions (Full Strategy)</b>",
            "<b>D) Strategy Comparison: Total Error Rates</b>"
        ),
        specs=[
            [{"type": "bar"}, {"type": "bar"}],
            [{"type": "bar"}, {"type": "bar"}]
        ],
        vertical_spacing=0.15,
        horizontal_spacing=0.12
    )
    
    # ========================================================================
    # PANEL A: Inter-hemispheric Error Rates (All Strategies)
    # ========================================================================
    
    strategies = ["Full Strategy", "LH Strategy", "RH Strategy"]
    conditions = ["Rest", "Task"]
    
    for condition in conditions:
        df_cond = df_metrics[df_metrics['Condition'] == condition]
        
        fig.add_trace(
            go.Bar(
                name=condition,
                x=df_cond['Strategy'],
                y=df_cond['Inter_Hemi_Pct'],
                text=[f"{val:.1f}%" for val in df_cond['Inter_Hemi_Pct']],
                textposition='outside',
                marker_color='lightcoral' if condition == 'Task' else 'steelblue',
                showlegend=True
            ),
            row=1, col=1
        )
    
    # ========================================================================
    # PANEL B: Hemispheric Asymmetry (L→R vs R→L)
    # ========================================================================
    
    # Focus on Full Strategy only for clarity
    df_full = df_metrics[df_metrics['Strategy'] == 'Full Strategy']
    
    x_labels = [f"{row['Condition']}" for _, row in df_full.iterrows()]
    
    fig.add_trace(
        go.Bar(
            name='Left → Right',
            x=x_labels,
            y=df_full['L2R_Pct'],
            text=[f"{val:.1f}%" for val in df_full['L2R_Pct']],
            textposition='outside',
            marker_color='rgb(99, 110, 250)',
            showlegend=True
        ),
        row=1, col=2
    )
    
    fig.add_trace(
        go.Bar(
            name='Right → Left',
            x=x_labels,
            y=df_full['R2L_Pct'],
            text=[f"{val:.1f}%" for val in df_full['R2L_Pct']],
            textposition='outside',
            marker_color='rgb(239, 85, 59)',
            showlegend=True
        ),
        row=1, col=2
    )
    
    # ========================================================================
    # PANEL C: Network Error Contributions (Full Strategy Only)
    # ========================================================================
    
    df_net_full = df_network[df_network['Strategy'] == 'Full Strategy']
    
    # Get unique networks and sort by average error contribution
    networks = df_net_full.groupby('Network')['Error_Pct'].mean().sort_values(ascending=False).index.tolist()
    
    for condition in conditions:
        df_cond = df_net_full[df_net_full['Condition'] == condition]
        
        # Ensure networks are in the same order
        df_cond = df_cond.set_index('Network').reindex(networks).reset_index()
        
        fig.add_trace(
            go.Bar(
                name=f"{condition}",
                x=df_cond['Network'],
                y=df_cond['Error_Pct'],
                text=[f"{val:.1f}%" for val in df_cond['Error_Pct']],
                textposition='outside',
                marker_color='lightcoral' if condition == 'Task' else 'steelblue',
                showlegend=False
            ),
            row=2, col=1
        )
    
    # ========================================================================
    # PANEL D: Strategy Comparison - Total Error Rates
    # ========================================================================
    
    # Calculate error rate as percentage of total possible misclassifications
    # (This is more meaningful than raw counts)
    
    for condition in conditions:
        df_cond = df_metrics[df_metrics['Condition'] == condition]
        
        fig.add_trace(
            go.Bar(
                name=f"{condition}",
                x=df_cond['Strategy'],
                y=df_cond['Total_Errors'],
                text=[f"{val}" for val in df_cond['Total_Errors']],
                textposition='outside',
                marker_color='lightcoral' if condition == 'Task' else 'steelblue',
                showlegend=False
            ),
            row=2, col=2
        )
    
    # ========================================================================
    # LAYOUT UPDATES
    # ========================================================================
    
    fig.update_xaxes(title_text="Strategy", row=1, col=1)
    fig.update_yaxes(title_text="Inter-Hemispheric Errors (%)", row=1, col=1)
    
    fig.update_xaxes(title_text="Condition", row=1, col=2)
    fig.update_yaxes(title_text="Directional Error Rate (%)", row=1, col=2)
    
    fig.update_xaxes(title_text="Brain Network", row=2, col=1, tickangle=-45)
    fig.update_yaxes(title_text="Error Contribution (%)", row=2, col=1)
    
    fig.update_xaxes(title_text="Strategy", row=2, col=2)
    fig.update_yaxes(title_text="Total Misclassifications", row=2, col=2)
    
    fig.update_layout(
        title_text="<b>Figure 2: Quantitative Analysis of Classification Error Patterns</b><br>" +
                   "<sup>Comparison Across Training Strategies and Testing Conditions</sup>",
        height=1000,
        width=1600,
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        barmode='group',
        font=dict(family="Arial, sans-serif", size=11),
        paper_bgcolor='white',
        plot_bgcolor='rgba(240, 240, 240, 0.5)'
    )
    
    # Add grid lines for better readability
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
    
    return fig, df_metrics, df_network

# ============================================================================
# 5. STATISTICAL SUMMARY GENERATION
# ============================================================================

def generate_summary_statistics(df_metrics, df_network):
    """Generate comprehensive statistical summary for thesis"""
    
    print("\n" + "="*80)
    print("FIGURE 2: COMPREHENSIVE STATISTICAL SUMMARY")
    print("="*80)
    
    # Key Finding 1: Inter-hemispheric error changes
    print("\n1. INTER-HEMISPHERIC ERROR PATTERNS")
    print("-" * 80)
    
    for strategy in df_metrics['Strategy'].unique():
        df_strat = df_metrics[df_metrics['Strategy'] == strategy]
        rest_inter = df_strat[df_strat['Condition'] == 'Rest']['Inter_Hemi_Pct'].values[0]
        task_inter = df_strat[df_strat['Condition'] == 'Task']['Inter_Hemi_Pct'].values[0]
        change = task_inter - rest_inter
        
        print(f"\n{strategy}:")
        print(f"  Rest: {rest_inter:.2f}%")
        print(f"  Task: {task_inter:.2f}%")
        print(f"  Change: {change:+.2f} pp {'(INCREASE)' if change > 0 else '(DECREASE)'}")
    
    # Key Finding 2: Hemispheric asymmetry
    print("\n\n2. HEMISPHERIC ASYMMETRY (Full Strategy)")
    print("-" * 80)
    
    df_full = df_metrics[df_metrics['Strategy'] == 'Full Strategy']
    for _, row in df_full.iterrows():
        asymmetry = row['L2R_Pct'] - row['R2L_Pct']
        dominant = "Left→Right" if asymmetry > 0 else "Right→Left"
        
        print(f"\n{row['Condition']}:")
        print(f"  L→R: {row['L2R_Pct']:.2f}%")
        print(f"  R→L: {row['R2L_Pct']:.2f}%")
        print(f"  Asymmetry: {abs(asymmetry):.2f} pp (Dominant: {dominant})")
    
    # Key Finding 3: Network vulnerability
    print("\n\n3. NETWORK-SPECIFIC VULNERABILITY (Full Strategy)")
    print("-" * 80)
    
    df_net_full = df_network[df_network['Strategy'] == 'Full Strategy']
    
    for condition in ['Rest', 'Task']:
        df_cond = df_net_full[df_net_full['Condition'] == condition]
        df_sorted = df_cond.sort_values('Error_Pct', ascending=False)
        
        print(f"\n{condition} - Top 3 Networks:")
        for i, (_, row) in enumerate(df_sorted.head(3).iterrows(), 1):
            print(f"  {i}. {row['Network']}: {row['Error_Pct']:.2f}% ({row['Error_Count']} errors)")
    
    # Key Finding 4: Strategy comparison
    print("\n\n4. STRATEGY PERFORMANCE COMPARISON")
    print("-" * 80)
    
    for condition in ['Rest', 'Task']:
        df_cond = df_metrics[df_metrics['Condition'] == condition]
        df_sorted = df_cond.sort_values('Total_Errors')
        
        print(f"\n{condition} - Total Errors by Strategy:")
        for _, row in df_sorted.iterrows():
            print(f"  {row['Strategy']}: {row['Total_Errors']} errors " +
                  f"({row['Inter_Hemi_Pct']:.1f}% inter-hemispheric)")
    
    # Key Finding 5: Rest→Task generalization
    print("\n\n5. REST→TASK GENERALIZATION DEGRADATION")
    print("-" * 80)
    
    for strategy in df_metrics['Strategy'].unique():
        df_strat = df_metrics[df_metrics['Strategy'] == strategy]
        rest_total = df_strat[df_strat['Condition'] == 'Rest']['Total_Errors'].values[0]
        task_total = df_strat[df_strat['Condition'] == 'Task']['Total_Errors'].values[0]
        
        change = task_total - rest_total
        pct_change = (change / rest_total) * 100 if rest_total > 0 else 0
        
        print(f"\n{strategy}:")
        print(f"  Error increase: {change:+d} ({pct_change:+.1f}%)")
    
    print("\n" + "="*80)

# ============================================================================
# EXECUTION
# ============================================================================

# Define your data mapping (same as before)
mapping = {
    ("Full Strategy", "Rest"): (confusion_matrix_cv_full, region_info_full),
    ("Full Strategy", "Task"): (confusion_matrix_task_full, region_info_full),
    ("LH Strategy", "Rest"):   (confusion_matrix_cv_lh, region_info_lh),
    ("LH Strategy", "Task"):   (confusion_matrix_task_lh, region_info_lh),
    ("RH Strategy", "Rest"):   (confusion_matrix_cv_rh, region_info_rh),
    ("RH Strategy", "Task"):   (confusion_matrix_task_rh, region_info_rh)
}

# Generate Figure 2
fig2, df_metrics, df_network = create_figure2_comparative(mapping)

fig2.show()

# Generate statistical summary
generate_summary_statistics(df_metrics, df_network)

# Optional: Save
# fig2.write_html("figure2_comparative_analysis.html")
# fig2.write_image("figure2_comparative_analysis.png", width=1600, height=1000, scale=2)

# Optional: Export data tables for thesis appendix
# df_metrics.to_csv("table_hemispheric_metrics.csv", index=False)
# df_network.to_csv("table_network_metrics.csv", index=False)

In [ ]:

# Networks 
# 1. Networks and Setup
networks = [
    'Visual', 'Somatomotor', 'Dorsal Attention', 'Salience/Ventral Attention',
    'Limbic', 'Control', 'Default', 'Subcortical'
]
rows = ['Full', 'Left', 'Right']
cols = ['Rest (CV)', 'Task (Test)', 'Difference (T-R)']

def aggregate_to_network_sum(region_matrix, region_info):
    net_labels = region_info['major_network'].values
    df = pd.DataFrame(region_matrix, index=net_labels, columns=net_labels)
    
    # Use .sum() to preserve absolute counts
    row_grouped = df.groupby(level=0).sum()
    full_grouped = row_grouped.transpose().groupby(level=0).sum().transpose()
    
    net_matrix = full_grouped.reindex(index=networks, columns=networks).fillna(0)
    return net_matrix.values

# 3. Process matrices for Absolute Counts
full_rest_net = aggregate_to_network_sum(confusion_matrix_cv_full, region_info_full)
full_task_net = aggregate_to_network_sum(confusion_matrix_task_full, region_info_full)
lh_rest_net = aggregate_to_network_sum(confusion_matrix_cv_lh, region_info_lh)
lh_task_net = aggregate_to_network_sum(confusion_matrix_task_lh, region_info_lh)
rh_rest_net = aggregate_to_network_sum(confusion_matrix_cv_rh, region_info_rh)
rh_task_net = aggregate_to_network_sum(confusion_matrix_task_rh, region_info_rh)

# 4. Organize for Plotting
matrix_data = [
    [full_rest_net, full_task_net, (full_task_net - full_rest_net)],
    [lh_rest_net,   lh_task_net,   (lh_task_net - lh_rest_net)],
    [rh_rest_net,   rh_task_net,   (rh_task_net - rh_rest_net)]
]
# 1. Colors from the Bar Chart theme
color_blue = '#2b8cbe'
color_orange = '#e6550d'

# 2. Define Custom Colorscales
colorscale_seq = [[0, '#f7fbff'], [1, color_blue]] 
colorscale_div = [[0, color_blue], [0.5, '#ffffff'], [1, color_orange]]

# 3. Create Heatmap Grid
# Increased vertical/horizontal spacing to prevent label overlap
fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=[f"<b>{r} Strategy: {c}</b>" for r in rows for c in cols],
    horizontal_spacing=0.12, 
    vertical_spacing=0.15
)

for i in range(3):
    for j in range(3):
        data = matrix_data[i][j].copy()
        mask = np.eye(len(networks), dtype=bool)
        data_masked = np.where(mask, None, data)
        
        if j < 2:
            colorscale = colorscale_seq
            zmin, zmax = 0, np.nanmax(data_masked[data_masked != None])
        else:
            colorscale = colorscale_div
            limit = np.nanmax(np.abs(data_masked[data_masked != None].astype(float)))
            zmin, zmax = -limit, limit

        fig.add_trace(
            go.Heatmap(
                z=data_masked,
                x=networks, 
                y=networks,
                colorscale=colorscale,
                zmin=zmin, 
                zmax=zmax,
                text=data_masked,
                texttemplate="%{text:.0f}",
                textfont={"size": 9},
                showscale=False,
                hovertemplate="True: %{y}<br>Predicted: %{x}<br>Value: %{z}<extra></extra>"
            ),
            row=i+1, col=j+1
        )

# 4. Final Layout Adjustments
fig.update_layout(
    title_text="<b>8-Network Absolute Off-Diagonal Error Counts </b>",
    template="plotly_white",
    height=1300, # Increased height to allow more room for x-labels
    width=1500,
    # Added explicit margins to prevent labels from cutting off at the edges
    margin=dict(l=150, r=50, t=150, b=150)
)

# 5. Prevent Overlap via tickangle and standoff
fig.update_xaxes(
    tickangle=45, 
    tickfont=dict(size=10),
    automargin=True # Automatically adjusts subplot size to fit labels
)

fig.update_yaxes(
    autorange="reversed", 
    tickfont=dict(size=10),
    automargin=True # Ensures network names don't overlap the y-axis
)

fig.show()

In [ ]:
# ============================================================================
# PART 1: PERCENTAGE CHANGE MATRICES
# ============================================================================

def calculate_percentage_change(rest_matrix, task_matrix):
    """
    Calculate percentage change: ((Task - Rest) / Rest) * 100
    Handle division by zero by setting to NaN
    """
    with np.errstate(divide='ignore', invalid='ignore'):
        pct_change = ((task_matrix - rest_matrix) / rest_matrix) * 100
        pct_change[~np.isfinite(pct_change)] = np.nan
    return pct_change

# Calculate percentage changes
full_pct = calculate_percentage_change(full_rest_net, full_task_net)
lh_pct = calculate_percentage_change(lh_rest_net, lh_task_net)
rh_pct = calculate_percentage_change(rh_rest_net, rh_task_net)

# Organize percentage change data
pct_matrix_data = [
    [full_rest_net, full_task_net, full_pct],
    [lh_rest_net,   lh_task_net,   lh_pct],
    [rh_rest_net,   rh_task_net,   rh_pct]
]

# Plot Percentage Change
rows = ['Full', 'LH_Hemi', 'RH_Hemi']
cols = ['Rest (Count)', 'Task (Count)', '% Change']

color_blue = '#2b8cbe'
color_orange = '#e6550d'
colorscale_seq = [[0, '#f7fbff'], [1, color_blue]]
colorscale_div = [[0, color_blue], [0.5, '#ffffff'], [1, color_orange]]

fig_pct = make_subplots(
    rows=3, cols=3,
    subplot_titles=[f"<b>{r}: {c}</b>" for r in rows for c in cols],
    horizontal_spacing=0.12,
    vertical_spacing=0.15
)

for i in range(3):
    for j in range(3):
        data = pct_matrix_data[i][j].copy()
        
        # Convert to float to accept NaN
        data_masked = data.astype(float)
        
        # Mask diagonal
        mask = np.eye(len(networks), dtype=bool)
        data_masked[mask] = np.nan
        
        if j < 2:  # Count data
            colorscale = colorscale_seq
            zmin, zmax = 0, np.nanmax(data_masked)
            text_template = "%{text:.0f}"
        else:  # Percentage change
            colorscale = colorscale_div
            valid_data = data_masked[~np.isnan(data_masked)]
            if len(valid_data) > 0:
                limit = np.nanpercentile(np.abs(valid_data), 95)
            else:
                limit = 100
            zmin, zmax = -limit, limit
            text_template = "%{text:.1f}%"

        fig_pct.add_trace(
            go.Heatmap(
                z=data_masked,
                x=networks,
                y=networks,
                colorscale=colorscale,
                zmin=zmin,
                zmax=zmax,
                text=data_masked,
                texttemplate=text_template,
                textfont={"size": 9},
                showscale=False,
                hovertemplate="<b>True:</b> %{y}<br><b>Predicted:</b> %{x}<br><b>Value:</b> %{text:.1f}<extra></extra>"
            ),
            row=i+1, col=j+1
        )

fig_pct.update_layout(
    title_text="<b>Network Classification: Absolute Counts and Percentage Change</b>",
    template="plotly_white",
    height=1400,
    width=1600,
    margin=dict(l=150, r=50, t=150, b=150)
)

fig_pct.update_xaxes(tickangle=45, tickfont=dict(size=10), automargin=True)
fig_pct.update_yaxes(autorange="reversed", tickfont=dict(size=10), automargin=True)

output_file_pct = VIZ_OUTPUT_DIR / 'network_percentage_change_analysis.html'
fig_pct.write_html(output_file_pct)
print(f"✓ Percentage change figure saved to: {output_file_pct}")
fig_pct.show()

# ============================================================================
# PART 2: STATISTICAL SIGNIFICANCE TESTING
# ============================================================================

def bootstrap_significance_test(rest_cm, task_cm, region_info, n_bootstrap=1000, alpha=0.05):
    """
    Bootstrap test for statistical significance of network-level changes.
    Tests if Task-Rest difference is significantly different from zero.
    """
    n_regions = rest_cm.shape[0]
    n_networks = len(networks)
    
    # Aggregate to network level
    rest_net = aggregate_to_network_sum(rest_cm, region_info)
    task_net = aggregate_to_network_sum(task_cm, region_info)
    observed_diff = task_net - rest_net
    
    # Bootstrap: resample regions within each network
    bootstrap_diffs = np.zeros((n_bootstrap, n_networks, n_networks))
    
    net_labels = region_info['major_network'].values
    
    for b in range(n_bootstrap):
        # Resample with replacement
        indices = np.random.choice(n_regions, size=n_regions, replace=True)
        
        # Create bootstrapped matrices
        boot_rest = rest_cm[indices][:, indices]
        boot_task = task_cm[indices][:, indices]
        boot_labels = net_labels[indices]
        
        # Create temporary region_info for bootstrap sample
        boot_region_info = pd.DataFrame({'major_network': boot_labels})
        
        # Aggregate
        boot_rest_net = aggregate_to_network_sum(boot_rest, boot_region_info)
        boot_task_net = aggregate_to_network_sum(boot_task, boot_region_info)
        
        bootstrap_diffs[b] = boot_task_net - boot_rest_net
    
    # Calculate p-values (two-tailed test)
    p_values = np.zeros((n_networks, n_networks))
    for i in range(n_networks):
        for j in range(n_networks):
            if i == j:  # Skip diagonal
                p_values[i, j] = 1.0
                continue
            
            # Two-tailed test
            boot_dist = bootstrap_diffs[:, i, j]
            obs_val = observed_diff[i, j]
            
            if obs_val > 0:
                p_val = np.mean(boot_dist <= 0) * 2
            else:
                p_val = np.mean(boot_dist >= 0) * 2
            
            p_values[i, j] = min(p_val, 1.0)
    
    # Significance mask (True = significant)
    significant = p_values < alpha
    
    return p_values, significant, observed_diff

print("\nPerforming bootstrap significance testing (this may take a minute)...")
print("Testing with 1000 bootstrap iterations...")

# Test all three models
p_values_full, sig_full, diff_full = bootstrap_significance_test(
    confusion_matrix_cv_full, confusion_matrix_task_full, region_info_full, n_bootstrap=1000
)

p_values_lh, sig_lh, diff_lh = bootstrap_significance_test(
    confusion_matrix_cv_lh, confusion_matrix_task_lh, region_info_lh, n_bootstrap=1000
)

p_values_rh, sig_rh, diff_rh = bootstrap_significance_test(
    confusion_matrix_cv_rh, confusion_matrix_task_rh, region_info_rh, n_bootstrap=1000
)

print("✓ Bootstrap testing complete!")

# Plot significance overlays
sig_matrix_data = [
    [diff_full, sig_full],
    [diff_lh, sig_lh],
    [diff_rh, sig_rh]
]

fig_sig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        '<b>Full: Difference</b>', '<b>Full: Significance (p<0.05)</b>',
        '<b>LH: Difference</b>', '<b>LH: Significance (p<0.05)</b>',
        '<b>RH: Difference</b>', '<b>RH: Significance (p<0.05)</b>'
    ],
    horizontal_spacing=0.12,
    vertical_spacing=0.15
)

for i in range(3):
    # Column 1: Difference values
    data_diff = sig_matrix_data[i][0].copy().astype(float)
    mask = np.eye(len(networks), dtype=bool)
    data_diff[mask] = np.nan
    
    valid_diff = data_diff[~np.isnan(data_diff)]
    if len(valid_diff) > 0:
        limit = np.nanpercentile(np.abs(valid_diff), 95)
    else:
        limit = 100
    
    fig_sig.add_trace(
        go.Heatmap(
            z=data_diff,
            x=networks,
            y=networks,
            colorscale=colorscale_div,
            zmin=-limit,
            zmax=limit,
            text=data_diff,
            texttemplate="%{text:.0f}",
            textfont={"size": 9},
            showscale=False,
            hovertemplate="<b>True:</b> %{y}<br><b>Predicted:</b> %{x}<br><b>Diff:</b> %{text:.1f}<extra></extra>"
        ),
        row=i+1, col=1
    )
    
    # Column 2: Significance (binary)
    sig_data = sig_matrix_data[i][1].astype(float)
    sig_data[mask] = np.nan
    
    # Create text annotations for significant cells
    text_annot = np.where(sig_matrix_data[i][1] & ~mask, '✓', '')
    
    fig_sig.add_trace(
        go.Heatmap(
            z=sig_data,
            x=networks,
            y=networks,
            colorscale=[[0, '#f0f0f0'], [1, '#d62728']],
            zmin=0,
            zmax=1,
            text=text_annot,
            texttemplate="%{text}",
            textfont={"size": 14, "color": "white"},
            showscale=False,
            hovertemplate="<b>True:</b> %{y}<br><b>Predicted:</b> %{x}<br><b>Significant:</b> %{z}<extra></extra>"
        ),
        row=i+1, col=2
    )

fig_sig.update_layout(
    title_text="<b>Statistical Significance of Task-Rest Differences (Bootstrap Test, p<0.05)</b>",
    template="plotly_white",
    height=1400,
    width=1400,
    margin=dict(l=150, r=50, t=150, b=150)
)

fig_sig.update_xaxes(tickangle=45, tickfont=dict(size=10), automargin=True)
fig_sig.update_yaxes(autorange="reversed", tickfont=dict(size=10), automargin=True)

output_file_sig = VIZ_OUTPUT_DIR / 'network_significance_analysis.html'
fig_sig.write_html(output_file_sig)
print(f"✓ Significance figure saved to: {output_file_sig}")
fig_sig.show()

# ============================================================================
# PART 3: NETWORK-LEVEL SUMMARY PLOTS
# ============================================================================

# Calculate summary statistics
def calculate_network_summaries(rest_net, task_net):
    """Calculate per-network summary statistics"""
    diff = task_net - rest_net
    
    # Mask diagonal
    mask = ~np.eye(len(networks), dtype=bool)
    
    summaries = []
    for i, net in enumerate(networks):
        # Errors FROM this network (misclassified as other networks)
        errors_from = diff[i, mask[i]].sum()
        
        # Errors TO this network (other networks misclassified as this)
        errors_to = diff[mask[:, i], i].sum()
        
        # Net change
        net_change = errors_to - errors_from
        
        summaries.append({
            'Network': net,
            'Errors_From': errors_from,
            'Errors_To': errors_to,
            'Net_Change': net_change,
            'Total_Change': np.abs(errors_from) + np.abs(errors_to)
        })
    
    return pd.DataFrame(summaries)

# Calculate for all models
summary_full = calculate_network_summaries(full_rest_net, full_task_net)
summary_lh = calculate_network_summaries(lh_rest_net, lh_task_net)
summary_rh = calculate_network_summaries(rh_rest_net, rh_task_net)

# Add model column
summary_full['Model'] = 'Full'
summary_lh['Model'] = 'LH'
summary_rh['Model'] = 'RH'

# Combine
summary_all = pd.concat([summary_full, summary_lh, summary_rh])

# Create summary plots
fig_summary = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        '<b>Total Change Magnitude</b>',
        '<b>Net Flow (Errors To - Errors From)</b>',
        '<b>Errors FROM Network</b>',
        '<b>Errors TO Network</b>'
    ],
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

models = ['Full', 'LH', 'RH']
colors = ['#2b8cbe', '#e6550d', '#31a354']

# Plot 1: Total Change
for idx, model in enumerate(models):
    df = summary_all[summary_all['Model'] == model]
    fig_summary.add_trace(
        go.Bar(
            name=model,
            x=df['Network'],
            y=df['Total_Change'],
            marker_color=colors[idx],
            text=df['Total_Change'],
            texttemplate='%{text:.0f}',
            textposition='outside'
        ),
        row=1, col=1
    )

# Plot 2: Net Change
for idx, model in enumerate(models):
    df = summary_all[summary_all['Model'] == model]
    fig_summary.add_trace(
        go.Bar(
            name=model,
            x=df['Network'],
            y=df['Net_Change'],
            marker_color=colors[idx],
            text=df['Net_Change'],
            texttemplate='%{text:.0f}',
            textposition='outside',
            showlegend=False
        ),
        row=1, col=2
    )

# Plot 3: Errors FROM
for idx, model in enumerate(models):
    df = summary_all[summary_all['Model'] == model]
    fig_summary.add_trace(
        go.Bar(
            name=model,
            x=df['Network'],
            y=df['Errors_From'],
            marker_color=colors[idx],
            text=df['Errors_From'],
            texttemplate='%{text:.0f}',
            textposition='outside',
            showlegend=False
        ),
        row=2, col=1
    )

# Plot 4: Errors TO
for idx, model in enumerate(models):
    df = summary_all[summary_all['Model'] == model]
    fig_summary.add_trace(
        go.Bar(
            name=model,
            x=df['Network'],
            y=df['Errors_To'],
            marker_color=colors[idx],
            text=df['Errors_To'],
            texttemplate='%{text:.0f}',
            textposition='outside',
            showlegend=False
        ),
        row=2, col=2
    )

fig_summary.update_layout(
    title_text="<b>Network-Level Summary: Task-Induced Changes</b>",
    template="plotly_white",
    height=1000,
    width=1600,
    barmode='group',
    showlegend=True,
    legend=dict(x=0.85, y=0.98)
)

fig_summary.update_xaxes(tickangle=45, tickfont=dict(size=10))
fig_summary.update_yaxes(title_text="Change in Errors", row=1, col=1)
fig_summary.update_yaxes(title_text="Net Change", row=1, col=2)
fig_summary.update_yaxes(title_text="Errors FROM", row=2, col=1)
fig_summary.update_yaxes(title_text="Errors TO", row=2, col=2)

# Add horizontal line at zero for net change plot
fig_summary.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=2)

output_file_summary = VIZ_OUTPUT_DIR / 'network_summary_analysis.html'
fig_summary.write_html(output_file_summary)
print(f"✓ Summary figure saved to: {output_file_summary}")
fig_summary.show()

# Print summary statistics
print("\n" + "="*80)
print("NETWORK SUMMARY STATISTICS")
print("="*80)
print(summary_all.to_string(index=False))

- Errors_From: 64  (Visual regions misclassified as other networks) 
- Errors_To: 30    (Other regions misclassified as Visual) 
- Net_Change: -34  (More leaving than entering) 

In [ ]:
# ===========================================================================
# 3-SUBPLOT VISUALIZATION: REST vs TASK ERROR RATES PER NETWORK
# ===========================================================================

print("\n" + "="*80)
print("CREATING REST vs TASK COMPARISON VISUALIZATIONS")
print("="*80)

# ===========================================================================
# Prepare data
# ===========================================================================

def prepare_network_error_data(cm_rest, cm_task, region_info_df, model_name):
    """
    Calculate error rates per network for rest and task
    
    Returns DataFrame with columns: Network, Rest_Error_Rate, Task_Error_Rate
    """
    networks = sorted(region_info_df['major_network'].unique())
    
    results = []
    
    for network in networks:
        # Get regions in this network
        network_mask = region_info_df['major_network'] == network
        network_indices = region_info_df[network_mask]['region_idx'].values
        
        # Calculate error rates for each region
        rest_errors = []
        task_errors = []
        
        for idx in network_indices:
            rest_total = cm_rest[idx, :].sum()
            task_total = cm_task[idx, :].sum()
            
            if rest_total > 0 and task_total > 0:
                rest_error = 1 - (cm_rest[idx, idx] / rest_total)
                task_error = 1 - (cm_task[idx, idx] / task_total)
                
                rest_errors.append(rest_error)
                task_errors.append(task_error)
        
        # Calculate mean error rates
        results.append({
            'Network': network,
            'Rest_Error_Rate': np.mean(rest_errors) if rest_errors else 0,
            'Task_Error_Rate': np.mean(task_errors) if task_errors else 0,
            'N_Regions': len(network_indices),
            'Model': model_name
        })
    
    return pd.DataFrame(results)

# Prepare data for all models
print("\nPreparing data for all models...")

data_full = prepare_network_error_data(
    confusion_matrix_cv_full, 
    confusion_matrix_task_full, 
    region_info_full,
    'Full Connectivity'
)

data_lh = prepare_network_error_data(
    confusion_matrix_cv_lh, 
    confusion_matrix_task_lh, 
    region_info_lh,
    'Left Hemisphere'
)

data_rh = prepare_network_error_data(
    confusion_matrix_cv_rh, 
    confusion_matrix_task_rh, 
    region_info_rh,
    'Right Hemisphere'
)

print("✓ Data prepared for all models")

# ===========================================================================
# Create 3-subplot figure
# ===========================================================================

print("\nCreating visualization...")

# Create subplots (1 row, 3 columns)
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        '<b>Full Connectivity (232 Regions)</b>',
        '<b>Left Hemisphere (116 Regions)</b>',
        '<b>Right Hemisphere (116 Regions)</b>'
    ],
    horizontal_spacing=0.08
)

# Color scheme
rest_color = '#2b8cbe'  # Blue. '#2b8cbe', '#e6550d'
task_color = '#e6550d'  # Purple/Magenta

# Sort networks by Full model task error rate for consistent ordering
network_order = data_full.sort_values('Task_Error_Rate', ascending=False)['Network'].tolist()

# ===========================================================================
# Subplot 1: Full Connectivity
# ===========================================================================

data_full_sorted = data_full.set_index('Network').loc[network_order].reset_index()

fig.add_trace(
    go.Bar(
        name='Rest (CV)',
        x=data_full_sorted['Network'],
        y=data_full_sorted['Rest_Error_Rate'],
        marker_color=rest_color,
        text=data_full_sorted['Rest_Error_Rate'].apply(lambda x: f'{x:.3f}'),
        textposition='outside',
        textfont=dict(size=9),
        showlegend=True,
        legendgroup='rest',
        offsetgroup=0
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        name='Task',
        x=data_full_sorted['Network'],
        y=data_full_sorted['Task_Error_Rate'],
        marker_color=task_color,
        text=data_full_sorted['Task_Error_Rate'].apply(lambda x: f'{x:.3f}'),
        textposition='outside',
        textfont=dict(size=9),
        showlegend=True,
        legendgroup='task',
        offsetgroup=1
    ),
    row=1, col=1
)

# ===========================================================================
# Subplot 2: Left Hemisphere
# ===========================================================================

data_lh_sorted = data_lh.set_index('Network').loc[network_order].reset_index()

fig.add_trace(
    go.Bar(
        name='Rest (CV)',
        x=data_lh_sorted['Network'],
        y=data_lh_sorted['Rest_Error_Rate'],
        marker_color=rest_color,
        text=data_lh_sorted['Rest_Error_Rate'].apply(lambda x: f'{x:.3f}'),
        textposition='outside',
        textfont=dict(size=9),
        showlegend=False,
        legendgroup='rest',
        offsetgroup=0
    ),
    row=1, col=2
)

fig.add_trace(
    go.Bar(
        name='Task',
        x=data_lh_sorted['Network'],
        y=data_lh_sorted['Task_Error_Rate'],
        marker_color=task_color,
        text=data_lh_sorted['Task_Error_Rate'].apply(lambda x: f'{x:.3f}'),
        textposition='outside',
        textfont=dict(size=9),
        showlegend=False,
        legendgroup='task',
        offsetgroup=1
    ),
    row=1, col=2
)

# ===========================================================================
# Subplot 3: Right Hemisphere
# ===========================================================================

data_rh_sorted = data_rh.set_index('Network').loc[network_order].reset_index()

fig.add_trace(
    go.Bar(
        name='Rest (CV)',
        x=data_rh_sorted['Network'],
        y=data_rh_sorted['Rest_Error_Rate'],
        marker_color=rest_color,
        text=data_rh_sorted['Rest_Error_Rate'].apply(lambda x: f'{x:.3f}'),
        textposition='outside',
        textfont=dict(size=9),
        showlegend=False,
        legendgroup='rest',
        offsetgroup=0
    ),
    row=1, col=3
)

fig.add_trace(
    go.Bar(
        name='Task',
        x=data_rh_sorted['Network'],
        y=data_rh_sorted['Task_Error_Rate'],
        marker_color=task_color,
        text=data_rh_sorted['Task_Error_Rate'].apply(lambda x: f'{x:.3f}'),
        textposition='outside',
        textfont=dict(size=9),
        showlegend=False,
        legendgroup='task',
        offsetgroup=1
    ),
    row=1, col=3
)

# ===========================================================================
# Update layout
# ===========================================================================

fig.update_layout(
    title_text="<b>Rest vs Task Error Rates by Network: Model Comparison</b>",
    template="plotly_white",
    height=700,
    width=1800,
    barmode='group',
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(size=14)
    )
)

# Update all x-axes
fig.update_xaxes(
    tickangle=45,
    tickfont=dict(size=10),
    title_text="Network",
    row=1, col=1
)
fig.update_xaxes(
    tickangle=45,
    tickfont=dict(size=10),
    title_text="Network",
    row=1, col=2
)
fig.update_xaxes(
    tickangle=45,
    tickfont=dict(size=10),
    title_text="Network",
    row=1, col=3
)

# Update all y-axes
fig.update_yaxes(
    title_text="Error Rate",
    range=[0, max(data_full['Task_Error_Rate'].max(), 
                  data_lh['Task_Error_Rate'].max(),
                  data_rh['Task_Error_Rate'].max()) * 1.15],
    row=1, col=1
)
fig.update_yaxes(
    title_text="Error Rate",
    range=[0, max(data_full['Task_Error_Rate'].max(), 
                  data_lh['Task_Error_Rate'].max(),
                  data_rh['Task_Error_Rate'].max()) * 1.15],
    row=1, col=2
)
fig.update_yaxes(
    title_text="Error Rate",
    range=[0, max(data_full['Task_Error_Rate'].max(), 
                  data_lh['Task_Error_Rate'].max(),
                  data_rh['Task_Error_Rate'].max()) * 1.15],
    row=1, col=3
)

# Save figure
output_file = VIZ_OUTPUT_DIR / 'rest_vs_task_comparison_3models.html'
fig.write_html(output_file)
print(f"✓ Visualization saved to: {output_file}")

fig.show()

print("\n✅ COMPLETE: 3-Subplot Rest vs Task Comparison")

In [ ]:
# ===========================================================================
# 3-SUBPLOT VISUALIZATION: REST vs TASK ERROR RATES PER NETWORK
# ===========================================================================

print("\n" + "="*80)
print("CREATING REST vs TASK COMPARISON VISUALIZATIONS")
print("="*80)

# ===========================================================================
# Prepare data
# ===========================================================================

def prepare_network_error_data(cm_rest, cm_task, region_info_df, model_name):
    """
    Calculate error rates per network for rest and task
    
    Returns DataFrame with columns: Network, Rest_Error_Rate, Task_Error_Rate
    """
    if region_info_df is None or cm_rest is None or cm_task is None:
        print(f"⚠️  Missing data for {model_name}")
        return pd.DataFrame()
    
    # Use 'network' column (not 'major_network')
    if 'network' not in region_info_df.columns:
        print(f"⚠️  'network' column not found in region_info for {model_name}")
        return pd.DataFrame()
    
    # Get unique networks, filtering out NaN values
    networks = region_info_df['network'].dropna().unique()
    networks = sorted([n for n in networks if isinstance(n, str)])
    
    results = []
    
    for network in networks:
        # Get regions in this network
        network_mask = region_info_df['network'] == network
        network_indices = region_info_df[network_mask]['region_idx'].values
        
        # Calculate error rates for each region
        rest_errors = []
        task_errors = []
        
        for idx in network_indices:
            # Check if index is within bounds
            if idx >= cm_rest.shape[0] or idx >= cm_task.shape[0]:
                continue
                
            rest_total = cm_rest[idx, :].sum()
            task_total = cm_task[idx, :].sum()
            
            if rest_total > 0 and task_total > 0:
                rest_error = 1 - (cm_rest[idx, idx] / rest_total)
                task_error = 1 - (cm_task[idx, idx] / task_total)
                
                rest_errors.append(rest_error)
                task_errors.append(task_error)
        
        # Calculate mean error rates
        if rest_errors and task_errors:
            results.append({
                'Network': network,
                'Rest_Error_Rate': np.mean(rest_errors),
                'Task_Error_Rate': np.mean(task_errors),
                'N_Regions': len(network_indices),
                'Model': model_name
            })
    
    return pd.DataFrame(results)

# Prepare data for all models
print("\nPreparing data for all models...")

data_full = prepare_network_error_data(
    confusion_matrix_cv_full, 
    confusion_matrix_task_full, 
    region_info_full,
    'Full Connectivity'
)
print(f"  Full Connectivity: {len(data_full)} networks")

data_lh = prepare_network_error_data(
    confusion_matrix_cv_lh, 
    confusion_matrix_task_lh, 
    region_info_lh,
    'Left Hemisphere'
)
print(f"  Left Hemisphere: {len(data_lh)} networks")

data_rh = prepare_network_error_data(
    confusion_matrix_cv_rh, 
    confusion_matrix_task_rh, 
    region_info_rh,
    'Right Hemisphere'
)
print(f"  Right Hemisphere: {len(data_rh)} networks")

print("✓ Data prepared for all models")

# Check if we have data
if data_full.empty and data_lh.empty and data_rh.empty:
    print("⚠️  No data available for visualization")
else:
    # ===========================================================================
    # Create 3-subplot figure
    # ===========================================================================
    
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    
    print("\nCreating visualization...")
    
    # Create subplots (1 row, 3 columns)
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=[
            '<b>Full Connectivity (232 Regions)</b>',
            '<b>Left Hemisphere (116 Regions)</b>',
            '<b>Right Hemisphere (116 Regions)</b>'
        ],
        horizontal_spacing=0.08
    )
    
    # Color scheme
    rest_color = '#2b8cbe'  # Blue
    task_color = '#e6550d'  # Orange
    
    # Sort networks by Full model task error rate for consistent ordering
    if not data_full.empty:
        network_order = data_full.sort_values('Task_Error_Rate', ascending=False)['Network'].tolist()
    elif not data_lh.empty:
        network_order = data_lh.sort_values('Task_Error_Rate', ascending=False)['Network'].tolist()
    else:
        network_order = data_rh.sort_values('Task_Error_Rate', ascending=False)['Network'].tolist()
    
    # ===========================================================================
    # Subplot 1: Full Connectivity
    # ===========================================================================
    
    if not data_full.empty:
        data_full_sorted = data_full.set_index('Network').reindex(network_order, fill_value=0).reset_index()
        
        fig.add_trace(
            go.Bar(
                name='Rest (CV)',
                x=data_full_sorted['Network'],
                y=data_full_sorted['Rest_Error_Rate'],
                marker_color=rest_color,
                text=data_full_sorted['Rest_Error_Rate'].apply(lambda x: f'{x:.3f}' if x > 0 else ''),
                textposition='outside',
                textfont=dict(size=9),
                showlegend=True,
                legendgroup='rest',
                offsetgroup=0
            ),
            row=1, col=1
        )
        
        fig.add_trace(
            go.Bar(
                name='Task',
                x=data_full_sorted['Network'],
                y=data_full_sorted['Task_Error_Rate'],
                marker_color=task_color,
                text=data_full_sorted['Task_Error_Rate'].apply(lambda x: f'{x:.3f}' if x > 0 else ''),
                textposition='outside',
                textfont=dict(size=9),
                showlegend=True,
                legendgroup='task',
                offsetgroup=1
            ),
            row=1, col=1
        )
    
    # ===========================================================================
    # Subplot 2: Left Hemisphere
    # ===========================================================================
    
    if not data_lh.empty:
        data_lh_sorted = data_lh.set_index('Network').reindex(network_order, fill_value=0).reset_index()
        
        fig.add_trace(
            go.Bar(
                name='Rest (CV)',
                x=data_lh_sorted['Network'],
                y=data_lh_sorted['Rest_Error_Rate'],
                marker_color=rest_color,
                text=data_lh_sorted['Rest_Error_Rate'].apply(lambda x: f'{x:.3f}' if x > 0 else ''),
                textposition='outside',
                textfont=dict(size=9),
                showlegend=False,
                legendgroup='rest',
                offsetgroup=0
            ),
            row=1, col=2
        )
        
        fig.add_trace(
            go.Bar(
                name='Task',
                x=data_lh_sorted['Network'],
                y=data_lh_sorted['Task_Error_Rate'],
                marker_color=task_color,
                text=data_lh_sorted['Task_Error_Rate'].apply(lambda x: f'{x:.3f}' if x > 0 else ''),
                textposition='outside',
                textfont=dict(size=9),
                showlegend=False,
                legendgroup='task',
                offsetgroup=1
            ),
            row=1, col=2
        )
    
    # ===========================================================================
    # Subplot 3: Right Hemisphere
    # ===========================================================================
    
    if not data_rh.empty:
        data_rh_sorted = data_rh.set_index('Network').reindex(network_order, fill_value=0).reset_index()
        
        fig.add_trace(
            go.Bar(
                name='Rest (CV)',
                x=data_rh_sorted['Network'],
                y=data_rh_sorted['Rest_Error_Rate'],
                marker_color=rest_color,
                text=data_rh_sorted['Rest_Error_Rate'].apply(lambda x: f'{x:.3f}' if x > 0 else ''),
                textposition='outside',
                textfont=dict(size=9),
                showlegend=False,
                legendgroup='rest',
                offsetgroup=0
            ),
            row=1, col=3
        )
        
        fig.add_trace(
            go.Bar(
                name='Task',
                x=data_rh_sorted['Network'],
                y=data_rh_sorted['Task_Error_Rate'],
                marker_color=task_color,
                text=data_rh_sorted['Task_Error_Rate'].apply(lambda x: f'{x:.3f}' if x > 0 else ''),
                textposition='outside',
                textfont=dict(size=9),
                showlegend=False,
                legendgroup='task',
                offsetgroup=1
            ),
            row=1, col=3
        )
    
    # ===========================================================================
    # Update layout
    # ===========================================================================
    
    # Calculate max y-axis value
    max_error = 0
    if not data_full.empty:
        max_error = max(max_error, data_full['Task_Error_Rate'].max())
    if not data_lh.empty:
        max_error = max(max_error, data_lh['Task_Error_Rate'].max())
    if not data_rh.empty:
        max_error = max(max_error, data_rh['Task_Error_Rate'].max())
    
    fig.update_layout(
        title_text="<b>Rest vs Task Error Rates by Network: Model Comparison</b>",
        template="plotly_white",
        height=700,
        width=1800,
        barmode='group',
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5,
            font=dict(size=14)
        )
    )
    
    # Update all x-axes
    for col in [1, 2, 3]:
        fig.update_xaxes(
            tickangle=45,
            tickfont=dict(size=10),
            title_text="Network",
            row=1, col=col
        )
    
    # Update all y-axes
    y_range = [0, max_error * 1.15]
    for col in [1, 2, 3]:
        fig.update_yaxes(
            title_text="Error Rate",
            range=y_range,
            row=1, col=col
        )
    
    # Save figure
    output_file = VIZ_OUTPUT_DIR / 'rest_vs_task_comparison_3models.html'
    fig.write_html(output_file)
    print(f"✓ Visualization saved to: {output_file}")
    
    fig.show()
    
    print("\n✅ COMPLETE: 3-Subplot Rest vs Task Comparison")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ===========================================================================
# RADAR (SPIDER) CHART: NETWORK ERROR PROFILES
# ===========================================================================

def create_network_radar_chart(data_frames, titles):
    """
    Create radar chart showing network error profiles across models.
    
    Args:
        data_frames: List of DataFrames with Network, Rest_Error_Rate, Task_Error_Rate
        titles: List of subplot titles
    
    Returns:
        Plotly figure
    """
    # Filter out empty dataframes
    valid_data = [(df, title) for df, title in zip(data_frames, titles) if not df.empty]
    
    if not valid_data:
        print("⚠️  No valid data for radar chart")
        return None
    
    n_plots = len(valid_data)
    
    # Create subplots with 'polar' type
    fig = make_subplots(
        rows=1, cols=n_plots,
        subplot_titles=[title for _, title in valid_data],
        specs=[[{'type': 'polar'} for _ in range(n_plots)]],
        horizontal_spacing=0.15
    )

    # Colors
    rest_color = 'rgba(43, 140, 190, 0.5)'  # Transparent Blue
    rest_line  = 'rgb(43, 140, 190)'
    task_color = 'rgba(230, 85, 13, 0.5)'   # Transparent Orange
    task_line  = 'rgb(230, 85, 13)'

    for i, (df, _) in enumerate(valid_data):
        # Radar charts must be "closed" loops, so we append the first row to the end
        df_closed = pd.concat([df, df.iloc[[0]]], ignore_index=True)
        networks = df_closed['Network'].tolist()
        
        # Add Rest Trace
        fig.add_trace(
            go.Scatterpolar(
                r=df_closed['Rest_Error_Rate'].tolist(),
                theta=networks,
                fill='toself',
                name='Rest (CV)',
                line=dict(color=rest_line, width=2),
                fillcolor=rest_color,
                legendgroup='rest',
                showlegend=True if i == 0 else False
            ),
            row=1, col=i+1
        )

        # Add Task Trace
        fig.add_trace(
            go.Scatterpolar(
                r=df_closed['Task_Error_Rate'].tolist(),
                theta=networks,
                fill='toself',
                name='Task',
                line=dict(color=task_line, width=2),
                fillcolor=task_color,
                legendgroup='task',
                showlegend=True if i == 0 else False
            ),
            row=1, col=i+1
        )

    # Global Max for consistent scaling across subplots
    max_val = max(df['Task_Error_Rate'].max() for df, _ in valid_data) * 1.15

    # Update Polar Layouts
    polar_settings = dict(
        radialaxis=dict(
            visible=True,
            range=[0, max_val],
            tickfont=dict(size=10),
            gridcolor="lightgrey",
            showticklabels=True
        ),
        angularaxis=dict(
            tickfont=dict(size=11),
            rotation=90,
            direction="clockwise"
        )
    )

    # Apply polar settings to each subplot
    layout_updates = {
        'title': "<b>Network Error Profiles: Rest vs. Task Fingerprints</b>",
        'height': 600,
        'width': 1600 if n_plots == 3 else 1200,
        'template': "plotly_white",
        'legend': dict(orientation="h", y=1.1, x=0.5, xanchor="center", font=dict(size=12))
    }
    
    for i in range(1, n_plots + 1):
        layout_updates[f'polar{i}'] = polar_settings
    
    fig.update_layout(**layout_updates)

    return fig

# ===========================================================================
# PREPARE SORTED DATA FOR RADAR CHART
# ===========================================================================

print("\n" + "="*80)
print("CREATING NETWORK RADAR CHART")
print("="*80)

# Determine network order (based on Full connectivity if available)
if not data_full.empty:
    network_order = data_full.sort_values('Task_Error_Rate', ascending=False)['Network'].tolist()
    print(f"Network order based on Full Connectivity ({len(network_order)} networks)")
elif not data_lh.empty:
    network_order = data_lh.sort_values('Task_Error_Rate', ascending=False)['Network'].tolist()
    print(f"Network order based on Left Hemisphere ({len(network_order)} networks)")
elif not data_rh.empty:
    network_order = data_rh.sort_values('Task_Error_Rate', ascending=False)['Network'].tolist()
    print(f"Network order based on Right Hemisphere ({len(network_order)} networks)")
else:
    print("⚠️  No data available for radar chart")
    network_order = []

# Create sorted dataframes
if network_order:
    # Full Connectivity
    if not data_full.empty:
        data_full_sorted = data_full.set_index('Network').reindex(network_order).dropna().reset_index()
    else:
        data_full_sorted = pd.DataFrame()
    
    # Left Hemisphere
    if not data_lh.empty:
        data_lh_sorted = data_lh.set_index('Network').reindex(network_order).dropna().reset_index()
    else:
        data_lh_sorted = pd.DataFrame()
    
    # Right Hemisphere
    if not data_rh.empty:
        data_rh_sorted = data_rh.set_index('Network').reindex(network_order).dropna().reset_index()
    else:
        data_rh_sorted = pd.DataFrame()
    
    print(f"✓ Sorted data prepared:")
    print(f"  Full: {len(data_full_sorted)} networks")
    print(f"  Left: {len(data_lh_sorted)} networks")
    print(f"  Right: {len(data_rh_sorted)} networks")
    
    # Create radar chart
    radar_data_list = [data_full_sorted, data_lh_sorted, data_rh_sorted]
    radar_titles = [
        '<b>Full Connectivity (232 Regions)</b>', 
        '<b>Left Hemisphere (116 Regions)</b>', 
        '<b>Right Hemisphere (116 Regions)</b>'
    ]
    
    print("\nGenerating radar chart...")
    radar_fig = create_network_radar_chart(radar_data_list, radar_titles)
    
    if radar_fig:
        # Save figure
        output_file = VIZ_OUTPUT_DIR / 'network_error_radar_chart.html'
        radar_fig.write_html(output_file)
        print(f"✓ Radar chart saved to: {output_file}")
        
        radar_fig.show()
        print("\n✅ COMPLETE: Network Radar Chart")
    else:
        print("⚠️  Could not create radar chart")
else:
    print("⚠️  No network data available for visualization")

### Region Level Analysis

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

# ===========================================================================
# VIOLIN PLOT: DISTRIBUTION OF REGION ERROR RATES
# ===========================================================================

print("\n" + "="*80)
print("CREATING REGION ERROR DISTRIBUTION VIOLIN PLOT")
print("="*80)

def analyze_region_performance(cm, region_info, model_label):
    """Calculates error rates for every individual ROI."""
    if cm is None or region_info is None:
        print(f"⚠️  Missing data for {model_label}")
        return pd.DataFrame()
    
    region_results = []
    
    for idx in range(len(region_info)):
        row = region_info.iloc[idx]
        
        # Check if index is within confusion matrix bounds
        if idx >= cm.shape[0]:
            continue
            
        total_samples = cm[idx, :].sum()
        
        if total_samples > 0:
            # Error = 1 - Recall (Sensitivity)
            error_rate = 1 - (cm[idx, idx] / total_samples)
        else:
            error_rate = np.nan
        
        # Use 'network' column (not 'major_network')
        network_name = row.get('network', 'Unknown')
        hemisphere = row.get('hemisphere', 'Both')
        region_name = row.get('region_name', f'Region_{idx}')
        
        region_results.append({
            'Region_Name': region_name,
            'Network': network_name,
            'Error_Rate': error_rate,
            'Model': model_label,
            'Hemisphere': hemisphere
        })
    
    df = pd.DataFrame(region_results)
    # Remove NaN values
    df = df.dropna(subset=['Error_Rate'])
    
    return df

# Generate DataFrames for all regions
print("\nAnalyzing region performance...")

df_full_regions = analyze_region_performance(
    confusion_matrix_task_full, 
    region_info_full, 
    'Full (232)'
)
print(f"  Full Connectivity: {len(df_full_regions)} regions")

df_lh_regions = analyze_region_performance(
    confusion_matrix_task_lh, 
    region_info_lh, 
    'Left (116)'
)
print(f"  Left Hemisphere: {len(df_lh_regions)} regions")

df_rh_regions = analyze_region_performance(
    confusion_matrix_task_rh, 
    region_info_rh, 
    'Right (116)'
)
print(f"  Right Hemisphere: {len(df_rh_regions)} regions")

# Combine data for visualization
combined_regions = pd.concat([df_full_regions, df_lh_regions, df_rh_regions], ignore_index=True)

print(f"\n✓ Combined data: {len(combined_regions)} total regions")

if not combined_regions.empty:
    # Calculate summary statistics
    print("\nSummary Statistics:")
    summary = combined_regions.groupby('Model')['Error_Rate'].agg(['mean', 'median', 'std', 'min', 'max'])
    print(summary.to_string())
    
    # Create violin plot
    print("\nCreating violin plot...")
    
    fig = px.violin(
        combined_regions, 
        x='Model', 
        y='Error_Rate', 
        color='Model',
        box=True,           # Show quartile boxes
        points='all',       # Show every individual region as a dot
        hover_data=['Region_Name', 'Network', 'Hemisphere'],
        title="<b>Distribution of Error Rates Across Individual Regions (Task Data)</b>",
        color_discrete_sequence=['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, Orange, Green
    )
    
    fig.update_layout(
        template="plotly_white",
        yaxis_title="Error Rate (Task)",
        xaxis_title="Connectivity Model",
        showlegend=False,
        height=600,
        width=1000,
        font=dict(size=12)
    )
    
    # Add mean line for reference
    fig.add_hline(
        y=combined_regions['Error_Rate'].mean(), 
        line_dash="dash", 
        line_color="gray",
        annotation_text=f"Overall Mean: {combined_regions['Error_Rate'].mean():.3f}",
        annotation_position="right"
    )
    
    # Save figure
    output_file = VIZ_OUTPUT_DIR / 'region_error_distribution_violin.html'
    fig.write_html(output_file)
    print(f"✓ Violin plot saved to: {output_file}")
    
    fig.show()
    
    print("\n✅ COMPLETE: Region Error Distribution Violin Plot")
    
    # Optional: Show worst performing regions
    print("\n" + "="*80)
    print("TOP 10 WORST PERFORMING REGIONS (Highest Error Rates)")
    print("="*80)
    worst_regions = combined_regions.nlargest(10, 'Error_Rate')[
        ['Model', 'Region_Name', 'Network', 'Hemisphere', 'Error_Rate']
    ]
    print(worst_regions.to_string(index=False))
    
    # Save to CSV
    combined_regions.to_csv(ANALYSIS_OUTPUT_DIR / 'region_error_rates_task.csv', index=False)
    print(f"\n✓ Saved detailed results to: {ANALYSIS_OUTPUT_DIR / 'region_error_rates_task.csv'}")
    
else:
    print("⚠️  No data available for violin plot")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

# ===========================================================================
# 9-SUBPLOT COMPREHENSIVE ANALYSIS: REST vs TASK FLOW + STABILITY
# ===========================================================================

print("\n" + "="*80)
print("CREATING 9-SUBPLOT COMPREHENSIVE ANALYSIS")
print("="*80)

# Check if data is loaded
required_vars = {
    'confusion_matrix_cv_full': confusion_matrix_cv_full,
    'confusion_matrix_task_full': confusion_matrix_task_full,
    'confusion_matrix_cv_lh': confusion_matrix_cv_lh,
    'confusion_matrix_task_lh': confusion_matrix_task_lh,
    'confusion_matrix_cv_rh': confusion_matrix_cv_rh,
    'confusion_matrix_task_rh': confusion_matrix_task_rh,
    'region_info_full': region_info_full,
    'region_info_lh': region_info_lh,
    'region_info_rh': region_info_rh
}

missing = [name for name, var in required_vars.items() if var is None]
if missing:
    print(f"⚠️  Missing required data: {', '.join(missing)}")
    print("Please run the data loading code first!")
else:
    print("✓ All required data loaded")
    
    # Group your confusion matrices
    matrices_dict = {
        'full_rest': confusion_matrix_cv_full,
        'full_task': confusion_matrix_task_full,
        'lh_rest': confusion_matrix_cv_lh,
        'lh_task': confusion_matrix_task_lh,
        'rh_rest': confusion_matrix_cv_rh,
        'rh_task': confusion_matrix_task_rh
    }
    
    # Group your region metadata
    infos_dict = {
        'full': region_info_full,
        'lh': region_info_lh,
        'rh': region_info_rh
    }
    
    def create_comprehensive_9_subplot(matrices_dict, infos_dict):
        """
        Create 9-subplot figure showing Rest/Task flows and stability changes.
        
        Layout:
        Row 1: Full Brain - Rest Flow | Task Flow | Stability Change
        Row 2: Left Hemisphere - Rest Flow | Task Flow | Stability Change
        Row 3: Right Hemisphere - Rest Flow | Task Flow | Stability Change
        """
        # 1. DEFINE MASTER NETWORK ORDER (Strictly alphabetical from the Full Model)
        # Use 'network' column (not 'major_network')
        if 'network' not in infos_dict['full'].columns:
            print("⚠️  'network' column not found in region_info")
            return None
        
        master_networks = sorted(infos_dict['full']['network'].dropna().unique())
        n_nets = len(master_networks)
        
        print(f"\nNetworks found: {n_nets}")
        print(f"  {', '.join(master_networks)}")
        
        # Standard Research Palette (Yeo 7-network inspired)
        NETWORK_COLORS = {
            'Vis': '#782486',           # Purple
            'SomMot': '#4682B4',        # Blue
            'DorsAttn': '#00760E',      # Green
            'SalVentAttn': '#C43AFA',   # Violet
            'Limbic': '#DCF8A4',        # Cream
            'Cont': '#E69422',          # Orange
            'Default': '#CD3E4E',       # Red
        }
        
        master_colors = [NETWORK_COLORS.get(net, '#808080') for net in master_networks]
    
        fig = make_subplots(
            rows=3, cols=3,
            subplot_titles=(
                "Full: Rest Flow", "Full: Task Flow", "Full: Stability Change",
                "LH: Rest Flow", "LH: Task Flow", "LH: Stability Change",
                "RH: Rest Flow", "RH: Task Flow", "RH: Stability Change"
            ),
            specs=[[{"type": "sankey"}, {"type": "sankey"}, {"type": "xy"}],
                   [{"type": "sankey"}, {"type": "sankey"}, {"type": "xy"}],
                   [{"type": "sankey"}, {"type": "sankey"}, {"type": "xy"}]],
            horizontal_spacing=0.07,
            vertical_spacing=0.1
        )
    
        strategies = [('full', 1), ('lh', 2), ('rh', 3)]
    
        for model, row_idx in strategies:
            info = infos_dict[model]
            
            def process_sankey_data(cm):
                """Process confusion matrix into network-level flows."""
                # We use master_networks to ensure the matrix is always n_nets x n_nets
                net_cm = np.zeros((n_nets, n_nets))
                
                for i, net_i in enumerate(master_networks):
                    # Use 'network' column
                    indices_i = info[info['network'] == net_i]['region_idx'].values
                    
                    for j, net_j in enumerate(master_networks):
                        indices_j = info[info['network'] == net_j]['region_idx'].values
                        
                        if len(indices_i) > 0 and len(indices_j) > 0:
                            # Extract submatrix and sum
                            net_cm[i, j] = cm[np.ix_(indices_i, indices_j)].sum()
                
                # Total Errors per Network for Bar Chart (exclude diagonal)
                total_net_errors = net_cm.sum(axis=1) - np.diag(net_cm) 
                
                # Zero out diagonal for Sankey (we only show errors)
                np.fill_diagonal(net_cm, 0)
                
                # Build Sankey links
                src, tgt, vals, clrs = [], [], [], []
                for i in range(n_nets):
                    for j in range(n_nets):
                        if net_cm[i, j] > 0:
                            src.append(i)
                            tgt.append(j + n_nets)
                            vals.append(net_cm[i, j])
                            
                            # Convert hex to rgba with transparency
                            hex_c = master_colors[i].lstrip('#')
                            rgb = tuple(int(hex_c[k:k+2], 16) for k in (0, 2, 4))
                            clrs.append(f'rgba({rgb[0]}, {rgb[1]}, {rgb[2]}, 0.4)')
                
                return src, tgt, vals, clrs, total_net_errors
    
            # Extract data for Rest and Task
            print(f"\nProcessing {model.upper()}...")
            s_r, t_r, v_r, c_r, err_r = process_sankey_data(matrices_dict[f'{model}_rest'])
            s_t, t_t, v_t, c_t, err_t = process_sankey_data(matrices_dict[f'{model}_task'])
            
            print(f"  Rest errors: {len(v_r)} flows, total={sum(v_r):.0f}")
            print(f"  Task errors: {len(v_t)} flows, total={sum(v_t):.0f}")
    
            # Col 1: Rest Sankey
            fig.add_trace(go.Sankey(
                node=dict(
                    pad=12, 
                    thickness=15, 
                    line=dict(color="black", width=0.5),
                    label=master_networks + [" "]*n_nets, 
                    color=master_colors*2
                ),
                link=dict(
                    source=s_r, 
                    target=t_r, 
                    value=v_r, 
                    color=c_r
                ),
                arrangement="perpendicular"
            ), row=row_idx, col=1)
    
            # Col 2: Task Sankey
            fig.add_trace(go.Sankey(
                node=dict(
                    pad=12, 
                    thickness=15, 
                    line=dict(color="black", width=0.5),
                    label=master_networks + [" "]*n_nets, 
                    color=master_colors*2
                ),
                link=dict(
                    source=s_t, 
                    target=t_t, 
                    value=v_t, 
                    color=c_t
                ),
                arrangement="perpendicular"
            ), row=row_idx, col=2)
    
            # Col 3: Percentage Change Bar
            pct_change = ((err_t - err_r) / (err_r + 1e-9)) * 100
            
            fig.add_trace(go.Bar(
                x=master_networks,
                y=pct_change,
                marker_color=master_colors,
                text=[f"{v:.1f}%" for v in pct_change],
                textposition='auto',
                textfont=dict(size=10),
                showlegend=False,
                hovertemplate='<b>%{x}</b><br>Change: %{y:.1f}%<extra></extra>'
            ), row=row_idx, col=3)
    
        # Global Formatting
        fig.update_layout(
            title=dict(
                text="<b>State-Dependent Network Error Analysis: Comparative Flow & Stability</b>", 
                x=0.5, 
                font=dict(size=22, family="Arial")
            ),
            height=1300, 
            width=1800,
            paper_bgcolor='white',
            plot_bgcolor='rgba(240, 240, 240, 0.4)'
        )
        
        # Synchronized Axis Formatting for Bar Charts
        for row in [1, 2, 3]:
            fig.update_xaxes(
                tickangle=-45, 
                tickfont=dict(size=10), 
                row=row, 
                col=3
            )
            fig.update_yaxes(
                title_text="% Change in Errors<br>(Rest → Task)", 
                row=row, 
                col=3
            )
        
        return fig
    
    # Execute
    print("\nCreating visualization...")
    fig_9_subplot = create_comprehensive_9_subplot(matrices_dict, infos_dict)
    
    if fig_9_subplot:
        # Save figure
        output_file = VIZ_OUTPUT_DIR / 'comprehensive_9subplot_analysis.html'
        fig_9_subplot.write_html(output_file)
        print(f"\n✓ 9-subplot figure saved to: {output_file}")
        
        fig_9_subplot.show()
        print("\n✅ COMPLETE: 9-Subplot Comprehensive Analysis")
    else:
        print("⚠️  Could not create figure")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# ===========================================================================
# HEMISPHERE COMPARISON HEATMAPS: REST vs TASK vs DIFFERENCE
# ===========================================================================

print("\n" + "="*80)
print("CREATING HEMISPHERE COMPARISON HEATMAPS")
print("="*80)

def get_normalized_matrix(cm):
    """Normalize CM by row and mask the diagonal for research focus."""
    row_sums = cm.sum(axis=1, keepdims=True)
    norm_cm = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums!=0)
    np.fill_diagonal(norm_cm, np.nan)
    return norm_cm

def plot_hemisphere_comparison(matrices, region_infos, model_key, display_title):
    """
    Create 3-panel heatmap showing Rest, Task, and Difference confusion patterns.
    
    Args:
        matrices: Dict of confusion matrices
        region_infos: Dict of region info DataFrames
        model_key: Key for this model ('full', 'lh', 'rh')
        display_title: Display title for the figure
    """
    # Check if data exists
    if model_key not in region_infos or region_infos[model_key] is None:
        print(f"⚠️  Missing region_info for {model_key}")
        return None
    
    if f'{model_key}_rest' not in matrices or matrices[f'{model_key}_rest'] is None:
        print(f"⚠️  Missing rest matrix for {model_key}")
        return None
        
    if f'{model_key}_task' not in matrices or matrices[f'{model_key}_task'] is None:
        print(f"⚠️  Missing task matrix for {model_key}")
        return None
    
    # 1. Setup Subplots: Minimalist titles
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=("Rest (Baseline)", "Task (State)", "Difference (State Shift)"),
        horizontal_spacing=0.1
    )

    # 2. Data Prep & Standardized Scaling
    info = region_infos[model_key].copy()
    
    # Use 'network' column (not 'major_network')
    if 'network' not in info.columns:
        print(f"⚠️  'network' column not found for {model_key}")
        return None
    
    # Sort by network and region name for organized display
    info = info.sort_values(['network', 'region_name'])
    indices = info['region_idx'].values
    num_regions = len(indices)
    
    print(f"\n{display_title}:")
    print(f"  Regions: {num_regions}")
    print(f"  Networks: {info['network'].nunique()}")
    
    m_rest = get_normalized_matrix(matrices[f'{model_key}_rest'][np.ix_(indices, indices)])
    m_task = get_normalized_matrix(matrices[f'{model_key}_task'][np.ix_(indices, indices)])
    m_diff = m_rest - m_task
    
    # Calculate global Vmax for Rest/Task to ensure direct comparability
    global_vmax = np.nanpercentile(np.concatenate([m_rest.flatten(), m_task.flatten()]), 98)
    diff_vmax = np.nanpercentile(np.abs(m_diff.flatten()), 98)
    
    print(f"  Rest/Task scale: [0, {global_vmax:.3f}]")
    print(f"  Difference scale: [{-diff_vmax:.3f}, {diff_vmax:.3f}]")

    # 3. Add Heatmaps
    plot_configs = [
        (m_rest, 'Viridis', 0, global_vmax, 1, "Error Rate"),
        (m_task, 'Viridis', 0, global_vmax, 2, "Error Rate"),
        (m_diff, 'RdBu_r', -diff_vmax, diff_vmax, 3, "Δ Rate")
    ]

    for data, scale, zmin, zmax, col, cb_title in plot_configs:
        fig.add_trace(go.Heatmap(
            z=data, 
            colorscale=scale, 
            zmin=zmin, 
            zmax=zmax,
            showscale=True if col >= 2 else False,  # Consolidate bars
            colorbar=dict(
                title=dict(text=f"<b>{cb_title}</b>", side="top", font=dict(size=10)),
                thickness=12, 
                len=0.6, 
                y=0.5,
                x=0.63 if col == 2 else 1.02,
                ticks="outside"
            ) if col >= 2 else None,
            hoverongaps=False,
            hovertemplate='True: %{y}<br>Pred: %{x}<br>Rate: %{z:.3f}<extra></extra>'
        ), row=1, col=col)

        # --- Network Boundaries ---
        # Use 'network' column
        net_changes = info['network'].values[:-1] != info['network'].values[1:]
        net_bounds = [0] + list(np.where(net_changes)[0] + 1) + [len(info)]
        unique_nets = info['network'].unique()

        for i, net_name in enumerate(unique_nets):
            start, end = net_bounds[i], net_bounds[i+1]
            mid = (start + end) / 2
            
            # Y-Labels (Only leftmost)
            if col == 1:
                fig.add_annotation(
                    x=-0.02, y=mid, xref="x1", yref="y1",
                    text=f"{net_name}", showarrow=False, textangle=0,
                    xanchor='right', font=dict(size=10, family="Arial", color="#2c3e50")
                )
            
            # X-Labels (All, Vertical)
            fig.add_annotation(
                x=mid, y=num_regions + (num_regions * 0.03), 
                xref=f"x{col}", yref=f"y{col}",
                text=f"{net_name}", showarrow=False, textangle=-90,
                yanchor='top', font=dict(size=10, family="Arial", color="#2c3e50")
            )

        # Boundary Lines: Optimized for heatmap contrast
        for b in net_bounds:
            line_color = "rgba(255,255,255,0.2)" if col < 3 else "rgba(0,0,0,0.1)"
            fig.add_shape(
                type="line", 
                x0=b, x1=b, y0=0, y1=num_regions, 
                line=dict(color=line_color, width=1), 
                row=1, col=col
            )
            fig.add_shape(
                type="line", 
                x0=0, x1=num_regions, y0=b, y1=b, 
                line=dict(color=line_color, width=1), 
                row=1, col=col
            )

    # 4. Global Refinement: Square Lock & Typography
    fig.update_xaxes(showticklabels=False, scaleanchor="y", scaleratio=1, constrain='domain')
    fig.update_yaxes(showticklabels=False, autorange='reversed', constrain='domain')

    fig.update_layout(
        title=dict(
            text=f"<b>Regional Misclassification Fingerprints: {display_title}</b>",
            x=0.5, y=0.96, font=dict(size=22, family="Arial", color="#2c3e50")
        ),
        width=1400, height=650,
        paper_bgcolor='white', plot_bgcolor='white',
        margin=dict(l=180, r=80, t=110, b=180)
    )
    
    return fig

# --- Check if data is loaded ---
required_vars = [
    ('confusion_matrix_cv_full', confusion_matrix_cv_full),
    ('confusion_matrix_task_full', confusion_matrix_task_full),
    ('confusion_matrix_cv_lh', confusion_matrix_cv_lh),
    ('confusion_matrix_task_lh', confusion_matrix_task_lh),
    ('confusion_matrix_cv_rh', confusion_matrix_cv_rh),
    ('confusion_matrix_task_rh', confusion_matrix_task_rh),
    ('region_info_full', region_info_full),
    ('region_info_lh', region_info_lh),
    ('region_info_rh', region_info_rh)
]

missing = [name for name, var in required_vars if var is None]
if missing:
    print(f"⚠️  Missing required data: {', '.join(missing)}")
    print("Please run the data loading code first!")
else:
    print("✓ All required data loaded")
    
    # --- Execution ---
    # 1. Define the models to be visualized
    models_to_plot = [
        ('full', 'Full Brain (232 ROIs)'), 
        ('lh', 'Left Hemisphere (116 ROIs)'), 
        ('rh', 'Right Hemisphere (116 ROIs)')
    ]
    
    # 2. Map your existing confusion matrices to the dictionary expected by the function
    matrices_dict = {
        'full_rest': confusion_matrix_cv_full,
        'full_task': confusion_matrix_task_full,
        'lh_rest': confusion_matrix_cv_lh,
        'lh_task': confusion_matrix_task_lh,
        'rh_rest': confusion_matrix_cv_rh,
        'rh_task': confusion_matrix_task_rh
    }
    
    # 3. Map your region info DataFrames
    infos_dict = {
        'full': region_info_full,
        'lh': region_info_lh,
        'rh': region_info_rh
    }
    
    # 4. Execute the plotting loop
    for key, title in models_to_plot:
        print(f"\nCreating heatmap for {title}...")
        figure = plot_hemisphere_comparison(matrices_dict, infos_dict, key, title)
        
        if figure:
            # Save figure
            output_file = VIZ_OUTPUT_DIR / f'heatmap_comparison_{key}.html'
            figure.write_html(output_file)
            print(f"✓ Saved to: {output_file}")
            
            figure.show()
        else:
            print(f"⚠️  Could not create figure for {key}")
    
    print("\n✅ COMPLETE: Hemisphere Comparison Heatmaps")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# ===========================================================================
# COMBINED RADAR PLOT: NETWORK STABILITY FINGERPRINTS
# ===========================================================================

print("\n" + "="*80)
print("CREATING COMBINED NETWORK RADAR PLOT")
print("="*80)

def create_combined_radar_plot(matrices, region_infos, global_max_r):
    """
    Create 3-panel radar plot showing network error rates for Rest vs Task.
    
    Args:
        matrices: Dict of confusion matrices
        region_infos: Dict of region info DataFrames
        global_max_r: Maximum radius for consistent scaling
    """
    # 1. Initialize Subplots with polar specification
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=("Full Brain (232)", "Left Hemisphere (116)", "Right Hemisphere (116)"),
        specs=[[{'type': 'polar'}, {'type': 'polar'}, {'type': 'polar'}]],
        horizontal_spacing=0.1
    )

    models = [('full', 1), ('lh', 2), ('rh', 3)]

    def get_network_errors(cm, info):
        """Calculate error rate per network."""
        if cm is None or info is None:
            return [], []
        
        # Use 'network' column (not 'major_network')
        if 'network' not in info.columns:
            print("⚠️  'network' column not found")
            return [], []
        
        networks = sorted(info['network'].dropna().unique())
        net_error_rates = []
        
        for net in networks:
            idx = info[info['network'] == net]['region_idx'].values
            
            if len(idx) == 0:
                continue
            
            # Check bounds
            if max(idx) >= cm.shape[0]:
                print(f"⚠️  Index out of bounds for network {net}")
                continue
            
            sub_cm = cm[np.ix_(idx, idx)]
            total_samples = sub_cm.sum()
            correct_samples = np.trace(sub_cm)
            error_rate = (total_samples - correct_samples) / total_samples if total_samples > 0 else 0
            net_error_rates.append(error_rate)
        
        return networks, net_error_rates

    for model_key, col_idx in models:
        # Check if data exists
        if model_key not in region_infos or region_infos[model_key] is None:
            print(f"⚠️  Missing region_info for {model_key}")
            continue
        
        if f'{model_key}_rest' not in matrices or matrices[f'{model_key}_rest'] is None:
            print(f"⚠️  Missing rest matrix for {model_key}")
            continue
        
        if f'{model_key}_task' not in matrices or matrices[f'{model_key}_task'] is None:
            print(f"⚠️  Missing task matrix for {model_key}")
            continue
        
        # Data Preparation
        info = region_infos[model_key]
        nets, rest_errors = get_network_errors(matrices[f'{model_key}_rest'], info)
        _, task_errors = get_network_errors(matrices[f'{model_key}_task'], info)
        
        if len(nets) == 0:
            print(f"⚠️  No networks found for {model_key}")
            continue
        
        print(f"\n{model_key.upper()}:")
        print(f"  Networks: {len(nets)}")
        print(f"  Rest error range: [{min(rest_errors):.3f}, {max(rest_errors):.3f}]")
        print(f"  Task error range: [{min(task_errors):.3f}, {max(task_errors):.3f}]")

        # Close loops for radar geometry
        nets_loop = nets + [nets[0]]
        rest_loop = rest_errors + [rest_errors[0]]
        task_loop = task_errors + [task_errors[0]]

        # 2. Add Traces to specific subplots
        fig.add_trace(go.Scatterpolar(
            r=rest_loop, 
            theta=nets_loop, 
            fill='toself', 
            name='Rest (CV)',
            line=dict(color='#2b8cbe', width=2.5),  # Blue
            fillcolor='rgba(43, 140, 190, 0.2)',
            legendgroup='rest', 
            showlegend=(col_idx == 1),
            hovertemplate='<b>%{theta}</b><br>Rest Error: %{r:.3f}<extra></extra>'
        ), row=1, col=col_idx)

        fig.add_trace(go.Scatterpolar(
            r=task_loop, 
            theta=nets_loop, 
            fill='toself', 
            name='Task',
            line=dict(color='#e6550d', width=2.5),  # Orange
            fillcolor='rgba(230, 85, 13, 0.2)',
            legendgroup='task', 
            showlegend=(col_idx == 1),
            hovertemplate='<b>%{theta}</b><br>Task Error: %{r:.3f}<extra></extra>'
        ), row=1, col=col_idx)

    # 3. Synchronized Academic Styling
    polar_settings = dict(
        bgcolor="white",
        radialaxis=dict(
            visible=True, 
            range=[0, global_max_r], 
            tickformat=".2f",
            gridcolor="rgba(0,0,0,0.1)", 
            tickfont=dict(size=10),
            showline=True,
            linecolor="rgba(0,0,0,0.2)"
        ),
        angularaxis=dict(
            gridcolor="rgba(0,0,0,0.1)", 
            rotation=90, 
            direction="clockwise",
            tickfont=dict(size=11)
        )
    )

    fig.update_layout(
        title=dict(
            text="<b>Comparative Network Stability Fingerprints: Rest vs Task</b>", 
            x=0.5, 
            font=dict(size=22, family="Arial")
        ),
        polar=polar_settings,
        polar2=polar_settings,
        polar3=polar_settings,
        width=1600, 
        height=650,
        paper_bgcolor='white',
        legend=dict(
            orientation="h", 
            y=-0.12, 
            x=0.5, 
            xanchor="center",
            font=dict(size=13)
        ),
        margin=dict(t=100, b=100, l=80, r=80)
    )

    return fig

# --- Check if data is loaded ---
required_vars = [
    ('confusion_matrix_cv_full', confusion_matrix_cv_full),
    ('confusion_matrix_task_full', confusion_matrix_task_full),
    ('confusion_matrix_cv_lh', confusion_matrix_cv_lh),
    ('confusion_matrix_task_lh', confusion_matrix_task_lh),
    ('confusion_matrix_cv_rh', confusion_matrix_cv_rh),
    ('confusion_matrix_task_rh', confusion_matrix_task_rh),
    ('region_info_full', region_info_full),
    ('region_info_lh', region_info_lh),
    ('region_info_rh', region_info_rh)
]

missing = [name for name, var in required_vars if var is None]
if missing:
    print(f"⚠️  Missing required data: {', '.join(missing)}")
    print("Please run the data loading code first!")
else:
    print("✓ All required data loaded")
    
    # --- Create dictionaries ---
    matrices_dict = {
        'full_rest': confusion_matrix_cv_full,
        'full_task': confusion_matrix_task_full,
        'lh_rest': confusion_matrix_cv_lh,
        'lh_task': confusion_matrix_task_lh,
        'rh_rest': confusion_matrix_cv_rh,
        'rh_task': confusion_matrix_task_rh
    }
    
    infos_dict = {
        'full': region_info_full,
        'lh': region_info_lh,
        'rh': region_info_rh
    }
    
    # Calculate appropriate global max from data
    print("\nCalculating global maximum error rate...")
    all_errors = []
    
    for key in ['full', 'lh', 'rh']:
        info = infos_dict[key]
        if info is not None and 'network' in info.columns:
            for matrix_key in [f'{key}_rest', f'{key}_task']:
                cm = matrices_dict.get(matrix_key)
                if cm is not None:
                    networks = info['network'].dropna().unique()
                    for net in networks:
                        idx = info[info['network'] == net]['region_idx'].values
                        if len(idx) > 0 and max(idx) < cm.shape[0]:
                            sub_cm = cm[np.ix_(idx, idx)]
                            total = sub_cm.sum()
                            correct = np.trace(sub_cm)
                            if total > 0:
                                all_errors.append((total - correct) / total)
    
    if all_errors:
        global_max_r = np.percentile(all_errors, 95) * 1.15  # 95th percentile + 15% margin
        print(f"✓ Global max error rate: {global_max_r:.3f}")
    else:
        global_max_r = 0.30  # Fallback
        print(f"⚠️  Using default global max: {global_max_r:.3f}")
    
    # --- Execution ---
    print("\nCreating combined radar plot...")
    combined_radar = create_combined_radar_plot(matrices_dict, infos_dict, global_max_r)
    
    if combined_radar:
        # Save figure
        output_file = VIZ_OUTPUT_DIR / 'combined_network_radar_rest_vs_task.html'
        combined_radar.write_html(output_file)
        print(f"\n✓ Saved to: {output_file}")
        
        combined_radar.show()
        print("\n✅ COMPLETE: Combined Network Radar Plot")
    else:
        print("⚠️  Could not create figure")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.patches import Patch
import matplotlib.colors as mcolors

# ===========================================================================
# PUBLICATION-QUALITY MATPLOTLIB FIGURE
# ===========================================================================

print("\n" + "="*80)
print("CREATING PUBLICATION FIGURE (MATPLOTLIB)")
print("="*80)

# ============================================================================
# STABLE STYLE & FONT SETTINGS
# ============================================================================
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Liberation Sans', 'sans-serif'],
    'pdf.fonttype': 42,
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
    'axes.titlesize': 14,
    'axes.labelsize': 11,
})

# Yeo Network Colors
YEO_COLORS = {
    'Vis': '#782486', 
    'SomMot': '#4682B4', 
    'DorsAttn': '#00760E', 
    'SalVentAttn': '#C43AFA', 
    'Limbic': '#DCF8A4', 
    'Cont': '#E69422', 
    'Default': '#CD3E4E', 
    'Subcortical': '#333333'
}

def get_intensity_color(color_hex, amount=0.4):
    """Mixes a network color with white for Rest intensity shading."""
    c = mcolors.to_rgb(color_hex)
    return [(1 - amount) + amount * x for x in c]

def extract_true_within_network_errors(cm, region_info, networks, hemi_code=None):
    """
    Extract within-network error counts (off-diagonal within network blocks).
    
    Args:
        cm: Confusion matrix
        region_info: DataFrame with region metadata
        networks: List of network names
        hemi_code: 'L', 'R', or None for full brain
    
    Returns:
        Dict mapping network names to error counts
    """
    errors = {}
    
    for net in networks:
        # Filter by hemisphere if specified
        if hemi_code:
            mask = (region_info['network'] == net) & (region_info['hemisphere'] == hemi_code.lower())
        else:
            mask = (region_info['network'] == net)
        
        idx = region_info[mask]['region_idx'].values
        
        if len(idx) == 0:
            errors[net] = 0
            continue
        
        # Extract sub-confusion matrix for this network
        sub_cm = cm[np.ix_(idx, idx)]
        
        # Total errors = sum - diagonal (within-network misclassifications)
        total_samples = sub_cm.sum()
        correct_samples = np.trace(sub_cm)
        errors[net] = total_samples - correct_samples
    
    return errors

def create_within_network_scatter_plot(cm_rest, cm_task, region_info, networks, hemi, ax, show_legend=False):
    """
    Create scatter plot showing within-network variance (Rest vs Task).
    """
    hemi_code = None if hemi == 'Full' else hemi[0]
    
    for net in networks:
        # Filter by hemisphere
        if hemi_code:
            mask = (region_info['network'] == net) & (region_info['hemisphere'] == hemi_code.lower())
        else:
            mask = (region_info['network'] == net)
        
        idx = region_info[mask]['region_idx'].values
        
        if len(idx) == 0:
            continue
        
        # Calculate error rates for each region
        rest_rates = []
        task_rates = []
        
        for i in idx:
            if i >= cm_rest.shape[0] or i >= cm_task.shape[0]:
                continue
            
            rest_total = cm_rest[i, :].sum()
            task_total = cm_task[i, :].sum()
            
            if rest_total > 0 and task_total > 0:
                rest_rate = 1 - (cm_rest[i, i] / rest_total)
                task_rate = 1 - (cm_task[i, i] / task_total)
                rest_rates.append(rest_rate)
                task_rates.append(task_rate)
        
        if rest_rates:
            ax.scatter(rest_rates, task_rates, 
                      color=YEO_COLORS.get(net, '#999999'),
                      s=80, alpha=0.7, edgecolors='black', linewidth=0.5,
                      label=net if show_legend else None)
    
    # Add diagonal reference line
    max_val = max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, linewidth=1, zorder=0)
    
    ax.set_xlabel('Rest Error Rate', fontweight='bold')
    ax.set_ylabel('Task Error Rate', fontweight='bold')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    sns.despine(ax=ax, offset=5)

def add_bar_labels(ax):
    """Add value labels on top of bars."""
    for container in ax.containers:
        ax.bar_label(container, fmt='%.0f', fontsize=8, padding=3)

def create_master_legend(fig, networks):
    """Create legend showing network colors."""
    legend_elements = [
        Patch(facecolor=YEO_COLORS.get(net, '#999999'), 
              edgecolor='black', linewidth=0.5, label=net)
        for net in networks
    ]
    
    # Add legend in the 4th column space
    fig.legend(handles=legend_elements, loc='center right', 
              bbox_to_anchor=(0.98, 0.5), frameon=True, 
              fontsize=11, title='Networks', title_fontsize=12)

def create_publication_figure(cm_rest, cm_task, region_info, networks):
    """
    Create comprehensive publication-quality figure.
    
    Rows:
    1. Scatter plots (Rest vs Task error rates per region)
    2. Magnitude bars (Total error counts per network)
    3. FRI bars (Functional Reorganization Index)
    
    Columns:
    1. Full Brain
    2. Left Hemisphere
    3. Right Hemisphere
    4. Legend
    """
    fig = plt.figure(figsize=(22, 16))
    # GS with 4th column for legend
    gs = fig.add_gridspec(3, 4, width_ratios=[1, 1, 1, 0.25], 
                         wspace=0.35, hspace=0.5)
    hemispheres = ['Full', 'Left', 'Right']
    
    for row in range(3):
        for col, hemi in enumerate(hemispheres):
            ax = fig.add_subplot(gs[row, col])
            hemi_code = None if hemi == 'Full' else hemi[0]
            
            # --- ROW 1: SCATTER ---
            if row == 0:
                create_within_network_scatter_plot(
                    cm_rest, cm_task, region_info, networks, hemi, ax, 
                    show_legend=False
                )
                ax.set_title(f'{hemi} Brain Network Variance', pad=15)
                
            # --- ROW 2: MAGNITUDE (Total Error Box in Top-Left) ---
            elif row == 1:
                r_errs = extract_true_within_network_errors(
                    cm_rest, region_info, networks, hemi_code
                )
                t_errs = extract_true_within_network_errors(
                    cm_task, region_info, networks, hemi_code
                )
                
                # Global hemisphere statistics
                total_rest = sum(r_errs.values())
                total_task = sum(t_errs.values())
                
                x = np.arange(len(networks))
                width = 0.38
                
                for i, net in enumerate(networks):
                    base_color = YEO_COLORS.get(net, '#999999')
                    ax.bar(x[i] - width/2, r_errs[net], width, 
                          color=get_intensity_color(base_color, 0.4),
                          label='Rest' if i == 0 else None)
                    ax.bar(x[i] + width/2, t_errs[net], width, 
                          color=base_color,
                          label='Task' if i == 0 else None)
                
                # Error count box to top-left
                stats_box = f"ΣErrors\nRest: {int(total_rest)}\nTask: {int(total_task)}"
                ax.text(0.05, 0.95, stats_box, transform=ax.transAxes, 
                       fontsize=10, fontweight='bold',
                       verticalalignment='top', horizontalalignment='left',
                       bbox=dict(boxstyle='round,pad=0.5', facecolor='white', 
                                alpha=0.9, edgecolor='#CCCCCC', linewidth=1))
                
                ax.set_title(f'Error Magnitude: {hemi}')
                ax.set_xticks(x)
                ax.set_xticklabels([n[:6] for n in networks], rotation=30, ha='right')
                ax.set_ylabel('Total Error Count', fontweight='bold')
                ax.grid(False)
                add_bar_labels(ax)
                sns.despine(ax=ax, offset=5)

            # --- ROW 3: REORGANIZATION INDEX ---
            elif row == 2:
                r_e = extract_true_within_network_errors(
                    cm_rest, region_info, networks, hemi_code
                )
                t_e = extract_true_within_network_errors(
                    cm_task, region_info, networks, hemi_code
                )
                
                fri_vals = []
                for i, net in enumerate(networks):
                    denom = (t_e[net] + r_e[net])
                    val = (t_e[net] - r_e[net]) / denom if denom > 0 else 0
                    fri_vals.append(val)
                    ax.bar(i, val, color=YEO_COLORS.get(net, '#999999'), 
                          edgecolor='black', linewidth=0.5)
                
                ax.axhline(0, color='black', linewidth=1.2, zorder=3)
                ax.set_title(f'Reorganization Index (FRI): {hemi}')
                ax.set_xticks(range(len(networks)))
                ax.set_xticklabels([n[:6] for n in networks], rotation=30, ha='right')
                ax.set_ylabel('FRI (Normalized)', fontweight='bold')
                
                if fri_vals:
                    ax.set_ylim(min(fri_vals)-0.15, max(fri_vals)+0.15)
                
                ax.grid(False)
                add_bar_labels(ax)
                sns.despine(ax=ax, offset=5)

    fig.suptitle('Network Coherence Analysis: Rest vs Task', 
                 fontsize=22, fontweight='bold', y=0.98)
    create_master_legend(fig, networks)
    
    return fig

# --- Check if data is loaded ---
required_vars = [
    ('confusion_matrix_cv_full', confusion_matrix_cv_full),
    ('confusion_matrix_task_full', confusion_matrix_task_full),
    ('region_info_full', region_info_full)
]

missing = [name for name, var in required_vars if var is None]
if missing:
    print(f"⚠️  Missing required data: {', '.join(missing)}")
    print("Please run the data loading code first!")
else:
    print("✓ All required data loaded")
    
    # Create networks list from data
    if 'network' in region_info_full.columns:
        networks_list = sorted(region_info_full['network'].dropna().unique())
        print(f"✓ Found {len(networks_list)} networks: {', '.join(networks_list)}")
    else:
        print("⚠️  'network' column not found in region_info")
        networks_list = []
    
    if networks_list:
        print("\nCreating publication figure...")
        fig = create_publication_figure(
            confusion_matrix_cv_full, 
            confusion_matrix_task_full, 
            region_info_full, 
            networks_list
        )
        
        # Save figure
        output_file = VIZ_OUTPUT_DIR / 'publication_network_dynamics.png'
        plt.savefig(output_file, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"✓ Saved PNG to: {output_file}")
        
        # Also save as PDF
        output_pdf = VIZ_OUTPUT_DIR / 'publication_network_dynamics.pdf'
        plt.savefig(output_pdf, bbox_inches='tight', facecolor='white')
        print(f"✓ Saved PDF to: {output_pdf}")
        
        plt.show()
        print("\n✅ COMPLETE: Publication Figure")
    else:
        print("⚠️  Cannot create figure without network data")

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

def plot_top_error_hubs(matrices, region_info, model_label='Full'):
    """Identifies and visualizes the top 20 regions with the largest change in error rates."""
    
    def get_error_df(cm, info):
        # Calculate recall-based error per region
        row_sums = cm.sum(axis=1)
        # Handle cases where a region might have zero samples to avoid division by zero
        error_rates = np.divide(
            (row_sums - np.diag(cm)), 
            row_sums, 
            out=np.zeros_like(row_sums, dtype=float), 
            where=row_sums != 0
        )
        return pd.DataFrame({
            'Region': info['region_name'],
            'Network': info['major_network'],
            'Error_Rate': error_rates
        })

    # 1. Extract error rates for the specific model
    df_rest = get_error_df(matrices[f'{model_label.lower()}_rest'], region_info)
    df_task = get_error_df(matrices[f'{model_label.lower()}_task'], region_info)

    # 2. Calculate State-Dependent Stability Change (Delta)
    df_importance = df_rest.copy()
    df_importance['Stability_Change'] = df_task['Error_Rate'] - df_rest['Error_Rate']
    
    # Sort by absolute magnitude of change
    df_top = df_importance.sort_values(by='Stability_Change', key=abs, ascending=False).head(20)

    # 3. Create Research-Grade Horizontal Bar Chart
    fig = px.bar(
        df_top,
        x='Stability_Change',
        y='Region',
        color='Network',
        orientation='h',
        title=f"<b>Top 20 Anatomical Hubs: Stability Shift ({model_label})</b>",
        labels={'Stability_Change': 'Δ Error Rate (Task - Rest)'},
        # Use a qualitative palette consistent with brain network standards
        color_discrete_sequence=px.colors.qualitative.Prism 
    )

    fig.update_layout(
        template="plotly_white",
        # Sort regions so the most impacted are at the top
        yaxis={'categoryorder':'total ascending'}, 
        paper_bgcolor='white',
        plot_bgcolor='rgba(245, 245, 245, 0.5)', # Subtle light-grey plot area
        font=dict(family="Arial", size=12),
        margin=dict(l=250, r=50, t=80, b=50),
        showlegend=True,
        legend=dict(title="Functional Network", orientation="h", y=-0.2, x=0.5, xanchor="center")
    )
    
    # Add a zero-line to distinguish between increased and decreased stability
    fig.add_vline(x=0, line_width=2, line_dash="dash", line_color="black")
    
    return fig

# --- INITIALIZE DATA DICTIONARY TO FIX NAMEERROR ---
# Replace these with your actual variables from previous steps
matrices_dict = {
    'full_rest': confusion_matrix_cv_full,
    'full_task': confusion_matrix_task_full,
    'lh_rest': confusion_matrix_cv_lh,
    'lh_task': confusion_matrix_task_lh,
    'rh_rest': confusion_matrix_cv_rh,
    'rh_task': confusion_matrix_task_rh
}

# --- EXECUTE ---
fig_importance = plot_top_error_hubs(matrices_dict, region_info_full, model_label='Full')
fig_importance.show()

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# ===========================================================================
# SANKEY DIAGRAM: TOP CONFUSED REGIONS ERROR FLOW
# ===========================================================================

print("\n" + "="*80)
print("CREATING CONFUSION SANKEY DIAGRAMS")
print("="*80)

def get_top_confused_flow(cm, info, top_n=10):
    """
    Identifies top N regions with highest error rates and extracts targets.
    
    Args:
        cm: Confusion matrix
        info: Region info DataFrame
        top_n: Number of top confused regions to show
    
    Returns:
        labels, sources, targets, values for Sankey diagram
    """
    if cm is None or info is None:
        return [], [], [], []
    
    # Use 'network' column (not 'major_network')
    if 'network' not in info.columns or 'region_name' not in info.columns:
        print("⚠️  Missing required columns in region_info")
        return [], [], [], []
    
    # 1. Convert to float to avoid UFuncTypeError
    row_sums = cm.sum(axis=1).astype(float)
    err_cm = cm.copy().astype(float)
    np.fill_diagonal(err_cm, 0)
    
    region_errors = err_cm.sum(axis=1)
    
    # Corrected division: explicitly provide float64 output array
    error_rates = np.divide(
        region_errors, 
        row_sums, 
        out=np.zeros_like(row_sums, dtype=float), 
        where=row_sums != 0
    )
    
    # 2. Identify Top N most confused regions
    if len(error_rates) < top_n:
        top_n = len(error_rates)
    
    top_indices = np.argsort(error_rates)[-top_n:][::-1]  # Descending order
    
    sources, targets, values, labels = [], [], [], []
    region_names = info['region_name'].values
    network_names = info['network'].values  # Fixed column name
    
    # 3. Build Sankey Structure
    for idx in top_indices:
        if idx >= len(region_names):
            continue
            
        actual_name = f"{region_names[idx]} ({network_names[idx]})"
        if actual_name not in labels:
            labels.append(actual_name)
        
        # Find top 3 confusion targets for this region
        confusion_row = err_cm[idx, :]
        target_indices = np.argsort(confusion_row)[-3:][::-1]  # Top 3, descending
        
        for t_idx in target_indices:
            if t_idx >= len(region_names):
                continue
                
            if confusion_row[t_idx] > 0:
                target_name = f"Pred: {region_names[t_idx]}"
                if target_name not in labels:
                    labels.append(target_name)
                
                sources.append(labels.index(actual_name))
                targets.append(labels.index(target_name))
                values.append(int(confusion_row[t_idx]))
    
    return labels, sources, targets, values

def plot_confusion_sankey(matrices, infos, strategies):
    """
    Generates Sankey diagrams for top confused regions across strategies.
    
    Args:
        matrices: Dict of confusion matrices
        infos: Dict of region info DataFrames  
        strategies: List of (model_key, display_name) tuples
    """
    for model_key, display_name in strategies:
        print(f"\nCreating Sankey for {display_name}...")
        
        # Check if data exists
        if f'{model_key}_task' not in matrices or matrices[f'{model_key}_task'] is None:
            print(f"⚠️  Missing task matrix for {model_key}")
            continue
        
        if model_key not in infos or infos[model_key] is None:
            print(f"⚠️  Missing region info for {model_key}")
            continue
        
        cm = matrices[f'{model_key}_task']
        info = infos[model_key]
        
        labels, sources, targets, values = get_top_confused_flow(cm, info, top_n=10)
        
        if not labels:
            print(f"⚠️  No confusion flow data for {model_key}")
            continue
        
        print(f"  Found {len([l for l in labels if not l.startswith('Pred:')])} confused regions")
        print(f"  Total flows: {len(values)}")
        print(f"  Total errors: {sum(values)}")
        
        # Create figure
        fig = go.Figure(data=[go.Sankey(
            node=dict(
                pad=15, 
                thickness=20,
                line=dict(color="black", width=0.5),
                label=labels,
                color="rgba(31, 119, 180, 0.8)"
            ),
            link=dict(
                source=sources, 
                target=targets, 
                value=values,
                color="rgba(200, 200, 200, 0.4)",
                hovertemplate='%{source.label} → %{target.label}<br>Errors: %{value}<extra></extra>'
            )
        )])

        fig.update_layout(
            title=dict(
                text=f"<b>Error Flow: Top 10 Confused Regions ({display_name})</b>",
                x=0.5, 
                font=dict(size=18, family="Arial")
            ),
            font_size=11, 
            width=1200, 
            height=750,
            paper_bgcolor='white',
            margin=dict(l=20, r=20, t=80, b=20)
        )
        
        # Save figure
        output_file = VIZ_OUTPUT_DIR / f'confusion_sankey_{model_key}.html'
        fig.write_html(output_file)
        print(f"  ✓ Saved to: {output_file}")
        
        fig.show()

# --- Check if data is loaded ---
required_vars = [
    ('confusion_matrix_cv_full', confusion_matrix_cv_full),
    ('confusion_matrix_task_full', confusion_matrix_task_full),
    ('confusion_matrix_cv_lh', confusion_matrix_cv_lh),
    ('confusion_matrix_task_lh', confusion_matrix_task_lh),
    ('confusion_matrix_cv_rh', confusion_matrix_cv_rh),
    ('confusion_matrix_task_rh', confusion_matrix_task_rh),
    ('region_info_full', region_info_full),
    ('region_info_lh', region_info_lh),
    ('region_info_rh', region_info_rh)
]

missing = [name for name, var in required_vars if var is None]
if missing:
    print(f"⚠️  Missing required data: {', '.join(missing)}")
    print("Please run the data loading code first!")
else:
    print("✓ All required data loaded")
    
    # --- Create dictionaries ---
    matrices_dict = {
        'full_rest': confusion_matrix_cv_full,
        'full_task': confusion_matrix_task_full,
        'lh_rest': confusion_matrix_cv_lh,
        'lh_task': confusion_matrix_task_lh,
        'rh_rest': confusion_matrix_cv_rh,
        'rh_task': confusion_matrix_task_rh
    }
    
    infos_dict = {
        'full': region_info_full,
        'lh': region_info_lh,
        'rh': region_info_rh
    }
    
    # --- Execution ---
    strategies_to_analyze = [
        ('full', 'Full Brain (232 Regions)'), 
        ('lh', 'Left Hemisphere (116 Regions)'), 
        ('rh', 'Right Hemisphere (116 Regions)')
    ]
    
    plot_confusion_sankey(matrices_dict, infos_dict, strategies_to_analyze)
    
    print("\n✅ COMPLETE: Confusion Sankey Diagrams")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
import plotly.express as px

def plot_9_subplot_anatomical_hubs(matrices, region_infos):
    # 1. Setup Subplot Grid (3 Rows, 3 Columns)
    # Rows: Rest, Task, Difference (Task - Rest)
    fig = make_subplots(
        rows=3, cols=3,
        subplot_titles=(
            "Full: Rest", "LH: Rest", "RH: Rest",
            "Full: Task", "LH: Task", "RH: Task",
            "Full: Δ (Task-Rest)", "LH: Δ (Task-Rest)", "RH: Δ (Task-Rest)"
        ),
        horizontal_spacing=0.07,
        vertical_spacing=0.1,
        shared_yaxes=False # Difference scale will be different from absolute counts
    )

    strategies = [('full', 'Full Brain'), ('lh', 'Left Hemisphere'), ('rh', 'Right Hemisphere')]
    
    networks = sorted(region_infos['full']['major_network'].unique())
    colors = px.colors.qualitative.Safe
    color_map = {net: colors[i % len(colors)] for i, net in enumerate(networks)}

    # Global max for rows 1 & 2 (Abs counts)
    max_abs = 0
    # Global max for row 3 (Difference)
    max_diff = 0

    # 2. Pre-calculate scales
    for model_key, _ in strategies:
        # Rest and Task
        r_cm = matrices[f'{model_key}_rest'].astype(float)
        t_cm = matrices[f'{model_key}_task'].astype(float)
        np.fill_diagonal(r_cm, 0); np.fill_diagonal(t_cm, 0)
        
        r_err = r_cm.sum(axis=1)
        t_err = t_cm.sum(axis=1)
        d_err = t_err - r_err
        
        max_abs = max(max_abs, r_err.max(), t_err.max())
        max_diff = max(max_diff, d_err.max())

    # 3. Populate the Grid
    for col_idx, (model_key, _) in enumerate(strategies, start=1):
        info = region_infos[model_key]
        r_cm = matrices[f'{model_key}_rest'].astype(float)
        t_cm = matrices[f'{model_key}_task'].astype(float)
        np.fill_diagonal(r_cm, 0); np.fill_diagonal(t_cm, 0)
        
        r_err = r_cm.sum(axis=1)
        t_err = t_cm.sum(axis=1)
        d_err = t_err - r_err # The "Task Error Plot" (Change)

        state_data = [
            (r_err, 1, max_abs, "Rest"),
            (t_err, 2, max_abs, "Task"),
            (d_err, 3, max_diff, "Difference")
        ]

        for err_vals, row_idx, scale_ref, label in state_data:
            df_plot = pd.DataFrame({
                'Region': info['region_name'],
                'Network': info['major_network'],
                'Error_Count': err_vals,
                'X_Pos': np.arange(len(info))
            })

            for net in networks:
                net_df = df_plot[df_plot['Network'] == net]
                if net_df.empty: continue
                
                # Use absolute value for bubble size in the difference plot 
                # (in case some errors decreased, though unlikely in this dataset)
                bubble_size = np.abs(net_df['Error_Count'])
                
                fig.add_trace(
                    go.Scatter(
                        x=net_df['X_Pos'],
                        y=net_df['Error_Count'],
                        mode='markers',
                        name=net,
                        marker=dict(
                            size=bubble_size,
                            sizemode='area',
                            sizeref=2. * scale_ref / (30.**2),
                            sizemin=2,
                            color=color_map[net],
                            line=dict(width=0.4, color='white')
                        ),
                        text=net_df['Region'],
                        hovertemplate="<b>%{text}</b><br>Errors: %{y}<extra></extra>",
                        legendgroup=net,
                        showlegend=(row_idx == 1 and col_idx == 1) 
                    ),
                    row=row_idx, col=col_idx
                )

    # 4. Refine Layout
    fig.update_layout(
        title=dict(
            text="9-Subplot Anatomical Error Analysis: Strategy vs. State Comparison",
            x=0.5, font=dict(size=22)
        ),
        height=1300, width=1700,
        template="plotly_white",
        legend=dict(title="Functional Network", orientation="h", y=-0.05, x=0.5, xanchor="center"),
        paper_bgcolor='white'
    )

    # Global Axis Formatting
    fig.update_xaxes(showticklabels=False, title="Anatomical Axis")
    fig.update_yaxes(title_text="Abs. Errors", row=1, col=1)
    fig.update_yaxes(title_text="Abs. Errors", row=2, col=1)
    fig.update_yaxes(title_text="Δ Errors (T-R)", row=3, col=1)
    
    # Set uniform ranges for absolute rows vs difference row
    fig.update_yaxes(range=[-5, max_abs * 1.1], row=1); fig.update_yaxes(range=[-5, max_abs * 1.1], row=2)
    fig.update_yaxes(range=[min(0, d_err.min()) - 5, max_diff * 1.1], row=3)

    return fig

# --- Execution ---
fig_9_hubs = plot_9_subplot_anatomical_hubs(matrices_dict, infos_dict)
fig_9_hubs.show()